# SmartForm Encoder — ML Training Pipeline

**Combined notebook** with all 7 training pipeline stages.

| Section | Purpose |
|---|---|
| 1. Data Export | Download verified entries from API as training data |
| 2. Preprocessing | Compare image preprocessing methods for best OCR |
| 3. Layout Detection | Train YOLOv8 to detect field regions on forms |
| 4. OCR Fine-tuning | Fine-tune PaddleOCR on Philippine handwriting |
| 5. Field Mapping | Train LayoutLMv3 for field classification |
| 6. Post-Correction | Rule-based + T5 to fix common OCR errors |
| 7. Evaluation | Benchmark CER, WER, field accuracy per template |

> **Run sections independently** — each section can be run on its own after the shared setup (Steps 0-1).
> Sections 2-7 expect data in `sample_data/` (ground_truth.json + images).

In [35]:
# ============================================================
# Step 0: Mount Google Drive & Set Working Directory
# ============================================================
# All notebooks share a common Drive folder so outputs persist
# across sessions and are accessible from other notebooks.

from google.colab import drive
import os

drive.mount('/content/drive')

# Set this to your notebooks folder in Google Drive
WORK_DIR = '/content/drive/MyDrive/smart-form-encoder/notebooks'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/smart-form-encoder/notebooks
Contents: ['smartform_ml_pipeline.ipynb', 'training_data']


In [36]:
# ============================================================
# Step 1: Install ALL dependencies (merged from all sections)
# ============================================================
import os
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

# Core ML/DL
!pip install -q transformers torch datasets seqeval sentencepiece protobuf

# PaddleOCR (pinned to 2.x for stable API)
!pip install -q 'paddlepaddle<3.0.0' 'paddleocr<3.0.0'

# Computer Vision & Image Processing
!pip install -q opencv-python-headless Pillow scikit-image ultralytics

# Data & Evaluation
!pip install -q pandas numpy matplotlib editdistance pyyaml requests tqdm

# Remove langchain packages — paddlex's RAG features not needed for OCR
!pip uninstall -y langchain langchain-community langchain-core langchain-text-splitters 2>/dev/null || true

print('All dependencies installed successfully')

All dependencies installed successfully


---

# Section 1: Training Data Export

Download verified form entries from the API as labeled training data.

In [ ]:
# ============================================================
# Configuration — UPDATE THESE VALUES
# ============================================================

API_BASE_URL = "http://localhost:8000/api/v1"  # Your SmartForm API URL
ADMIN_EMAIL = "admin@smartform.dev"           # Admin account email
ADMIN_PASSWORD = "admin1234"                     # Admin account password

# Optional: filter by template
TEMPLATE_ID = None  # Set to a UUID string to filter, or None for all templates

# Export settings
MAX_ENTRIES = 5000     # Maximum entries to export
BATCH_SIZE = 100       # Entries per API request
OUTPUT_DIR = "./training_data"  # Local output directory
DOWNLOAD_IMAGES = True  # Whether to download form images

In [38]:
import json
import os
import time
from datetime import datetime
from pathlib import Path

import requests
from tqdm.auto import tqdm

# Create output directories
output_path = Path(OUTPUT_DIR)
images_path = output_path / "images"
output_path.mkdir(parents=True, exist_ok=True)
images_path.mkdir(exist_ok=True)

print(f"Output directory: {output_path.absolute()}")

Output directory: /content/drive/MyDrive/smart-form-encoder/notebooks/training_data


## Step 1: Authenticate with the API

In [39]:
def authenticate(base_url: str, email: str, password: str) -> str:
    """Login and return access token."""
    resp = requests.post(
        f"{base_url}/auth/login",
        json={"email": email, "password": password},
    )
    resp.raise_for_status()
    data = resp.json()
    token = data["data"]["access_token"]
    print(f"Authenticated as {email}")
    return token


token = authenticate(API_BASE_URL, ADMIN_EMAIL, ADMIN_PASSWORD)
headers = {"Authorization": f"Bearer {token}"}

Authenticated as admin@smartform.dev


## Step 2: Check Available Training Data

In [40]:
# Get training data statistics
stats_resp = requests.get(f"{API_BASE_URL}/ml/training-stats", headers=headers)
stats_resp.raise_for_status()
stats = stats_resp.json()["data"]

print("=" * 60)
print("TRAINING DATA STATISTICS")
print("=" * 60)
print(f"Total verified entries:    {stats['total_verified_entries']}")
print(f"Total fields:             {stats['total_fields']}")
print(f"Corrected fields:         {stats['total_corrected_fields']}")
print(f"Correction rate:          {stats['correction_rate']:.1f}%")
print(f"Average confidence:       {stats['avg_confidence']:.4f}")
print()
print("Per-template breakdown:")
for t in stats.get("templates", []):
    print(f"  - {t['template_name']}: {t['verified_entries']} entries")

TRAINING DATA STATISTICS
Total verified entries:    1
Total fields:             55
Corrected fields:         0
Correction rate:          0.0%
Average confidence:       0.9200

Per-template breakdown:
  - DTI Business Name Registration: 1 entries


## Step 3: Export Training Data

In [41]:
def fetch_training_data(
    base_url: str,
    headers: dict,
    template_id: str | None = None,
    max_entries: int = 5000,
    batch_size: int = 100,
) -> list[dict]:
    """Fetch all training data entries in batches."""
    all_entries = []
    offset = 0

    with tqdm(total=max_entries, desc="Fetching entries") as pbar:
        while offset < max_entries:
            params = {
                "limit": min(batch_size, max_entries - offset),
                "offset": offset,
            }
            if template_id:
                params["template_id"] = template_id

            resp = requests.get(
                f"{base_url}/ml/export-training-data",
                headers=headers,
                params=params,
            )
            resp.raise_for_status()
            data = resp.json()["data"]

            entries = data["entries"]
            if not entries:
                break

            all_entries.extend(entries)
            pbar.update(len(entries))
            offset += batch_size

            if len(entries) < batch_size:
                break

    print(f"\nTotal entries fetched: {len(all_entries)}")
    return all_entries


entries = fetch_training_data(
    API_BASE_URL, headers, TEMPLATE_ID, MAX_ENTRIES, BATCH_SIZE
)

Fetching entries:   0%|          | 0/5000 [00:00<?, ?it/s]


Total entries fetched: 1


## Step 4: Download Form Images

In [42]:
def download_images(entries: list[dict], images_dir: Path) -> dict[str, str]:
    """Download form images from pre-signed URLs. Returns {entry_id: local_path}."""
    image_map = {}
    failed = []

    for entry in tqdm(entries, desc="Downloading images"):
        entry_id = entry["entry_id"]
        image_url = entry["image_url"]

        if not image_url or image_url.startswith("forms/"):
            # Raw R2 key, not a presigned URL — skip
            image_map[entry_id] = image_url
            continue

        # Determine extension
        ext = ".jpg"
        if "png" in image_url.lower():
            ext = ".png"
        elif "pdf" in image_url.lower():
            ext = ".pdf"

        local_path = images_dir / f"{entry_id}{ext}"

        if local_path.exists():
            image_map[entry_id] = str(local_path)
            continue

        try:
            resp = requests.get(image_url, timeout=30)
            resp.raise_for_status()
            local_path.write_bytes(resp.content)
            image_map[entry_id] = str(local_path)
        except Exception as e:
            failed.append((entry_id, str(e)))
            image_map[entry_id] = None

    print(f"\nDownloaded: {len(image_map) - len(failed)} | Failed: {len(failed)}")
    if failed:
        print("Failed entries:")
        for eid, err in failed[:10]:
            print(f"  {eid}: {err}")

    return image_map


if DOWNLOAD_IMAGES:
    image_map = download_images(entries, images_path)
else:
    image_map = {e["entry_id"]: e["image_url"] for e in entries}
    print("Skipping image download (DOWNLOAD_IMAGES=False)")


Downloaded: 1 | Failed: 0


## Step 5: Save Labeled Dataset

In [43]:
# Build the labeled dataset
dataset = {
    "export_date": datetime.now().isoformat(),
    "total_entries": len(entries),
    "entries": [],
}

total_fields = 0
corrected_fields = 0

for entry in entries:
    entry_data = {
        "entry_id": entry["entry_id"],
        "template_id": entry["template_id"],
        "template_name": entry["template_name"],
        "local_image_path": image_map.get(entry["entry_id"]),
        "confidence_score": entry.get("confidence_score"),
        "fields": [],
    }

    for field in entry["fields"]:
        total_fields += 1
        if field["was_corrected"]:
            corrected_fields += 1

        entry_data["fields"].append({
            "field_name": field["field_name"],
            "ocr_value": field["ocr_value"],
            "verified_value": field["verified_value"],
            "was_corrected": field["was_corrected"],
            "confidence": field["confidence"],
        })

    dataset["entries"].append(entry_data)

# Save labels
labels_path = output_path / "labels.json"
with open(labels_path, "w") as f:
    json.dump(dataset, f, indent=2, default=str)

# Save metadata
metadata = {
    "export_date": datetime.now().isoformat(),
    "total_entries": len(entries),
    "total_fields": total_fields,
    "corrected_fields": corrected_fields,
    "correction_rate": round(corrected_fields / total_fields * 100, 2) if total_fields else 0,
    "template_filter": TEMPLATE_ID,
    "api_source": API_BASE_URL,
}
metadata_path = output_path / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Entries:           {len(entries)}")
print(f"Total fields:      {total_fields}")
print(f"Corrected fields:  {corrected_fields}")
print(f"Correction rate:   {metadata['correction_rate']:.1f}%")
print(f"Labels saved to:   {labels_path}")
print(f"Metadata saved to: {metadata_path}")
print(f"Images saved to:   {images_path}")

EXPORT COMPLETE
Entries:           1
Total fields:      55
Corrected fields:  0
Correction rate:   0.0%
Labels saved to:   training_data/labels.json
Metadata saved to: training_data/metadata.json
Images saved to:   training_data/images


## Step 6: Dataset Analysis

In [44]:
# Analyze the exported dataset
from collections import Counter

# Template distribution
template_counts = Counter(e["template_name"] for e in entries)
print("\nEntries per template:")
for name, count in template_counts.most_common():
    print(f"  {name}: {count}")

# Field correction distribution
field_corrections = Counter()
field_totals = Counter()
for entry in entries:
    for field in entry["fields"]:
        field_totals[field["field_name"]] += 1
        if field["was_corrected"]:
            field_corrections[field["field_name"]] += 1

print("\nMost frequently corrected fields:")
for field_name, count in field_corrections.most_common(20):
    total = field_totals[field_name]
    pct = count / total * 100
    print(f"  {field_name}: {count}/{total} ({pct:.1f}% correction rate)")

# Confidence distribution
confidences = [
    f["confidence"]
    for e in entries
    for f in e["fields"]
]
if confidences:
    import statistics
    print(f"\nConfidence distribution:")
    print(f"  Mean:   {statistics.mean(confidences):.4f}")
    print(f"  Median: {statistics.median(confidences):.4f}")
    print(f"  StdDev: {statistics.stdev(confidences):.4f}")
    print(f"  Min:    {min(confidences):.4f}")
    print(f"  Max:    {max(confidences):.4f}")


Entries per template:
  DTI Business Name Registration: 1

Most frequently corrected fields:

Confidence distribution:
  Mean:   0.9200
  Median: 0.9200
  StdDev: 0.0000
  Min:    0.9200
  Max:    0.9200


---

# Section 2: Preprocessing Experiments

Test and compare image preprocessing methods to find the best pipeline for Philippine handwritten forms.

In [45]:
# ============================================================
# Step 2: Imports & Configuration
# ============================================================
import json
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
import editdistance
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (14, 8)

# Paths
SAMPLE_DIR = Path("./sample_data")
IMAGE_DIR = SAMPLE_DIR / "images"
GROUND_TRUTH_PATH = SAMPLE_DIR / "ground_truth.json"

# Load ground truth
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

print(f"Loaded ground truth with {len(ground_truth['entries'])} entries")
print(f"First entry image: {ground_truth['entries'][0]['image_file']}")
print(f"Fields per entry: {len(ground_truth['entries'][0]['fields'])}")

FileNotFoundError: [Errno 2] No such file or directory: 'sample_data/ground_truth.json'

In [ ]:
# ============================================================
# Step 3: Initialize PaddleOCR
# ============================================================
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=False, show_log=False)
print("PaddleOCR initialized (CPU mode)")

In [ ]:
# ============================================================
# Step 4: Define Preprocessing Functions
# ============================================================

def preprocess_original(img: np.ndarray) -> np.ndarray:
    """No preprocessing — baseline."""
    return img


def preprocess_grayscale(img: np.ndarray) -> np.ndarray:
    """Convert to grayscale (3-channel for PaddleOCR compatibility)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)


def preprocess_denoise_bilateral(img: np.ndarray) -> np.ndarray:
    """Bilateral filter — preserves edges while reducing noise."""
    return cv2.bilateralFilter(img, 9, 75, 75)


def preprocess_denoise_nlm(img: np.ndarray) -> np.ndarray:
    """Non-local means denoising — stronger denoising."""
    return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


def preprocess_binarize_otsu(img: np.ndarray) -> np.ndarray:
    """Otsu's thresholding — good for consistent lighting."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)


def preprocess_binarize_adaptive(img: np.ndarray) -> np.ndarray:
    """Adaptive thresholding — handles uneven lighting."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)


def preprocess_clahe(img: np.ndarray) -> np.ndarray:
    """CLAHE contrast enhancement in LAB color space."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l_channel)
    enhanced = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


def preprocess_sharpen(img: np.ndarray) -> np.ndarray:
    """Unsharp mask sharpening — enhances text edges."""
    gaussian = cv2.GaussianBlur(img, (0, 0), 3)
    sharpened = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
    return sharpened


def preprocess_deskew(img: np.ndarray) -> np.ndarray:
    """Correct skew angle from scanning/photography."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 10:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.5:  # Skip tiny corrections
        return img
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )


def preprocess_morphology_clean(img: np.ndarray) -> np.ndarray:
    """Morphological operations to clean noise from scanned forms."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Remove small noise dots
    kernel = np.ones((2, 2), np.uint8)
    cleaned = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel)
    # Close small gaps in text
    kernel = np.ones((2, 2), np.uint8)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
    return cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)


def preprocess_combined_light(img: np.ndarray) -> np.ndarray:
    """Light combo: denoise + CLAHE + sharpen."""
    img = preprocess_denoise_bilateral(img)
    img = preprocess_clahe(img)
    img = preprocess_sharpen(img)
    return img


def preprocess_combined_heavy(img: np.ndarray) -> np.ndarray:
    """Heavy combo: deskew + NLM denoise + CLAHE + sharpen."""
    img = preprocess_deskew(img)
    img = preprocess_denoise_nlm(img)
    img = preprocess_clahe(img)
    img = preprocess_sharpen(img)
    return img


def preprocess_scan_optimized(img: np.ndarray) -> np.ndarray:
    """Optimized for scanned documents: deskew + bilateral + CLAHE."""
    img = preprocess_deskew(img)
    img = preprocess_denoise_bilateral(img)
    img = preprocess_clahe(img)
    return img


# Registry of all preprocessing methods to test
PREPROCESSING_METHODS = {
    "original": preprocess_original,
    "grayscale": preprocess_grayscale,
    "denoise_bilateral": preprocess_denoise_bilateral,
    "denoise_nlm": preprocess_denoise_nlm,
    "binarize_otsu": preprocess_binarize_otsu,
    "binarize_adaptive": preprocess_binarize_adaptive,
    "clahe": preprocess_clahe,
    "sharpen": preprocess_sharpen,
    "deskew": preprocess_deskew,
    "morphology_clean": preprocess_morphology_clean,
    "combined_light": preprocess_combined_light,
    "combined_heavy": preprocess_combined_heavy,
    "scan_optimized": preprocess_scan_optimized,
}

print(f"Registered {len(PREPROCESSING_METHODS)} preprocessing methods")

In [ ]:
# ============================================================
# Step 5: OCR & Evaluation Helpers
# ============================================================

def run_ocr(img_array: np.ndarray) -> list[dict]:
    """Run PaddleOCR and return list of {text, confidence, bbox}."""
    result = ocr.ocr(img_array)
    lines = []
    if result and result[0]:
        for line in result[0]:
            lines.append({
                "text": line[1][0],
                "confidence": float(line[1][1]),
                "bbox": line[0],
            })
    return lines


def compute_cer(ocr_text: str, reference: str) -> float:
    """Character Error Rate = edit_distance(ocr, ref) / len(ref)."""
    if not reference:
        return 0.0 if not ocr_text else 1.0
    return editdistance.eval(ocr_text, reference) / len(reference)


def compute_wer(ocr_text: str, reference: str) -> float:
    """Word Error Rate = word_edit_distance(ocr, ref) / len(ref_words)."""
    ref_words = reference.split()
    ocr_words = ocr_text.split()
    if not ref_words:
        return 0.0 if not ocr_words else 1.0
    return editdistance.eval(ocr_words, ref_words) / len(ref_words)


def find_field_in_ocr(field_value: str, ocr_lines: list[dict], threshold: float = 0.7) -> dict | None:
    """Find the best matching OCR line for a ground truth field value.
    Uses character-level similarity with a threshold."""
    if not field_value:
        return None
    best_match = None
    best_score = 0
    for line in ocr_lines:
        ocr_text = line["text"].lower().strip()
        gt_text = field_value.lower().strip()
        # Exact containment
        if gt_text in ocr_text or ocr_text in gt_text:
            return line
        # Similarity score
        max_len = max(len(ocr_text), len(gt_text))
        if max_len == 0:
            continue
        dist = editdistance.eval(ocr_text, gt_text)
        score = 1 - (dist / max_len)
        if score > best_score:
            best_score = score
            best_match = line
    if best_score >= threshold:
        return best_match
    return None


print("Evaluation helpers ready")

In [ ]:
# ============================================================
# Step 6: Run OCR on Each Preprocessing Variant
# ============================================================

results = {}

for entry in ground_truth["entries"]:
    img_path = SAMPLE_DIR / entry["image_file"]
    if not img_path.exists():
        print(f"  Skipping {img_path} (file not found)")
        continue

    img = cv2.imread(str(img_path))
    print(f"\nProcessing: {img_path.name} ({img.shape[1]}x{img.shape[0]})")
    print("-" * 100)

    gt_fields = [f for f in entry["fields"] if f["type"] == "text" and f["verified_value"]]
    gt_text = " ".join(f["verified_value"] for f in gt_fields)

    for method_name, preprocess_fn in PREPROCESSING_METHODS.items():
        # Preprocess
        start = time.time()
        processed = preprocess_fn(img.copy())
        preprocess_time = time.time() - start

        # OCR
        start = time.time()
        ocr_lines = run_ocr(processed)
        ocr_time = time.time() - start

        # Aggregate OCR text
        full_text = " ".join(l["text"] for l in ocr_lines)
        avg_conf = np.mean([l["confidence"] for l in ocr_lines]) if ocr_lines else 0.0

        # Full-text CER/WER
        cer = compute_cer(full_text, gt_text)
        wer = compute_wer(full_text, gt_text)

        # Field-level matching: how many GT values found in OCR output
        fields_found = 0
        field_cer_sum = 0
        for field in gt_fields:
            match = find_field_in_ocr(field["verified_value"], ocr_lines, threshold=0.5)
            if match:
                fields_found += 1
                field_cer_sum += compute_cer(
                    match["text"].lower().strip(),
                    field["verified_value"].lower().strip(),
                )

        field_recall = fields_found / len(gt_fields) * 100 if gt_fields else 0
        avg_field_cer = (field_cer_sum / fields_found * 100) if fields_found else 100

        results[method_name] = {
            "lines_detected": len(ocr_lines),
            "avg_confidence": round(avg_conf, 4),
            "cer": round(cer * 100, 2),
            "wer": round(wer * 100, 2),
            "field_recall_pct": round(field_recall, 2),
            "avg_field_cer_pct": round(avg_field_cer, 2),
            "preprocess_ms": round(preprocess_time * 1000, 1),
            "ocr_time_s": round(ocr_time, 2),
        }

        print(
            f"  {method_name:25s} | lines: {len(ocr_lines):3d} | "
            f"conf: {avg_conf:.3f} | CER: {cer*100:5.1f}% | "
            f"WER: {wer*100:5.1f}% | recall: {field_recall:5.1f}% | "
            f"fieldCER: {avg_field_cer:5.1f}% | time: {ocr_time:.1f}s"
        )

print("\nDone! All preprocessing methods evaluated.")

In [ ]:
# ============================================================
# Step 7: Results Table & Ranking
# ============================================================
import pandas as pd

df = pd.DataFrame(results).T
df.index.name = "method"

# Sort by field recall (desc), then by avg field CER (asc)
df = df.sort_values(["field_recall_pct", "avg_field_cer_pct"], ascending=[False, True])

print("\n" + "=" * 110)
print("PREPROCESSING COMPARISON — FULL RESULTS")
print("=" * 110)
print(df.to_string())

best = df.index[0]
print(f"\n{'='*60}")
print(f"BEST METHOD: {best}")
print(f"  Field Recall:     {df.loc[best, 'field_recall_pct']}%")
print(f"  Avg Field CER:    {df.loc[best, 'avg_field_cer_pct']}%")
print(f"  Full-text CER:    {df.loc[best, 'cer']}%")
print(f"  Full-text WER:    {df.loc[best, 'wer']}%")
print(f"  Avg Confidence:   {df.loc[best, 'avg_confidence']}")
print(f"  Lines Detected:   {int(df.loc[best, 'lines_detected'])}")
print(f"  Preprocess Time:  {df.loc[best, 'preprocess_ms']}ms")
print(f"  OCR Time:         {df.loc[best, 'ocr_time_s']}s")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Step 8: Visualization Charts
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

df["field_recall_pct"].plot(kind="barh", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Field Recall (% GT fields found)")
axes[0, 0].set_xlabel("%")

df["avg_field_cer_pct"].plot(kind="barh", ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Avg Field CER (lower = better)")
axes[0, 1].set_xlabel("%")

df["cer"].plot(kind="barh", ax=axes[0, 2], color="salmon")
axes[0, 2].set_title("Full-text CER (lower = better)")
axes[0, 2].set_xlabel("%")

df["avg_confidence"].plot(kind="barh", ax=axes[1, 0], color="mediumseagreen")
axes[1, 0].set_title("Average OCR Confidence")
axes[1, 0].set_xlabel("Score (0-1)")

df["lines_detected"].plot(kind="barh", ax=axes[1, 1], color="goldenrod")
axes[1, 1].set_title("Lines Detected by OCR")
axes[1, 1].set_xlabel("Count")

df["ocr_time_s"].plot(kind="barh", ax=axes[1, 2], color="mediumpurple")
axes[1, 2].set_title("OCR Processing Time")
axes[1, 2].set_xlabel("Seconds")

plt.suptitle("Preprocessing Methods Comparison", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("preprocessing_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: preprocessing_comparison.png")

In [ ]:
# ============================================================
# Step 9: Visual Comparison — Cropped Region
# ============================================================
entry = ground_truth["entries"][0]
img_path = SAMPLE_DIR / entry["image_file"]
img = cv2.imread(str(img_path))

# Crop an area likely containing handwritten text
h, w = img.shape[:2]
crop = img[int(h*0.15):int(h*0.35), 0:w]

n_methods = len(PREPROCESSING_METHODS)
cols = 4
rows = (n_methods + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
axes = axes.flatten()

for idx, (name, fn) in enumerate(PREPROCESSING_METHODS.items()):
    processed = fn(crop.copy())
    rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(rgb)
    axes[idx].set_title(name, fontsize=10, fontweight="bold")
    axes[idx].axis("off")

for idx in range(n_methods, len(axes)):
    axes[idx].axis("off")

plt.suptitle("Visual Preprocessing Comparison (Cropped Region)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("preprocessing_visual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: preprocessing_visual.png")

In [ ]:
# ============================================================
# Step 10: Detailed Field-by-Field Analysis (Best Method)
# ============================================================
best_method = df.index[0]
best_fn = PREPROCESSING_METHODS[best_method]

entry = ground_truth["entries"][0]
img_path = SAMPLE_DIR / entry["image_file"]
img = cv2.imread(str(img_path))
processed = best_fn(img.copy())
ocr_lines = run_ocr(processed)

print(f"Field-by-field analysis using '{best_method}' preprocessing")
print("=" * 90)
print(f"{'Field Name':35s} | {'Ground Truth':25s} | {'OCR Match':25s} | Match")
print("-" * 90)

gt_fields = [f for f in entry["fields"] if f["type"] == "text" and f["verified_value"]]
matched = 0
for field in gt_fields:
    match = find_field_in_ocr(field["verified_value"], ocr_lines, threshold=0.5)
    status = "YES" if match else "NO"
    ocr_val = match["text"][:25] if match else "—"
    gt_val = field["verified_value"][:25]
    if match:
        matched += 1
    print(f"  {field['field_name']:35s} | {gt_val:25s} | {ocr_val:25s} | {status}")

print(f"\nMatched: {matched}/{len(gt_fields)} ({matched/len(gt_fields)*100:.1f}%)")

In [ ]:
# ============================================================
# Step 11: Save Preprocessing Configuration
# ============================================================
config = {
    "best_method": best_method,
    "ranking": list(df.index),
    "metrics": {
        method: {
            "field_recall_pct": float(results[method]["field_recall_pct"]),
            "cer": float(results[method].get("cer", 0)),
            "wer": float(results[method].get("wer", 0)),
        }
        for method in results
    },
}

# Save config JSON
config_path = SAMPLE_DIR / "preprocessing_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"\nBest preprocessing method: {best_method}")
print(f"Configuration saved to: {config_path}")
print(f"\nUse this in ocr_service.py:")
print(f"  PREPROCESSING_METHOD = '{best_method}'")

---

# Section 3: Layout Detection (YOLOv8)

Train a YOLOv8 object detection model to automatically detect field regions on form images.

In [ ]:
# ============================================================
# Step 2: Imports & Configuration
# ============================================================
import json
import os
import shutil
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml

# ---- Configuration ----
PROJECT_DIR = Path("./layout_detection")
DATASET_DIR = PROJECT_DIR / "dataset"
ANNOTATIONS_DIR = Path("./sample_data/annotations")
IMAGES_DIR = Path("./sample_data/images")
MODEL_OUTPUT_DIR = PROJECT_DIR / "runs"

# YOLO class definitions
CLASSES = [
    "text_field",
    "text_area",
    "checkbox",
    "signature",
    "date_field",
    "header",
]
NUM_CLASSES = len(CLASSES)

# Training config
EPOCHS = 100
IMAGE_SIZE = 640  # YOLOv8 default
BATCH_SIZE = 8    # Adjust for Colab T4 GPU
MODEL_SIZE = "yolov8s"  # s=small (good balance of speed/accuracy)

print(f"Classes: {CLASSES}")
print(f"Training: {EPOCHS} epochs, {IMAGE_SIZE}px, batch={BATCH_SIZE}")

In [ ]:
# ============================================================
# Step 3: Create YOLO Dataset Structure
# ============================================================

def create_dataset_structure():
    """Create the standard YOLO dataset directory structure."""
    for split in ["train", "val", "test"]:
        (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    print(f"Dataset structure created at {DATASET_DIR}")


def create_yolo_yaml():
    """Create the YOLO dataset configuration YAML file."""
    config = {
        "path": str(DATASET_DIR.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": NUM_CLASSES,
        "names": CLASSES,
    }
    yaml_path = DATASET_DIR / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    print(f"YOLO config saved to {yaml_path}")
    return yaml_path


create_dataset_structure()
yaml_path = create_yolo_yaml()

In [ ]:
# ============================================================
# Step 4: Manual Annotation Helper
# ============================================================
# For MVP with limited images, we manually define bounding boxes.
# For production, use Label Studio or Roboflow.

def create_manual_annotations(image_path: str, fields: list[dict]) -> list[dict]:
    """Convert ground truth field locations to YOLO format annotations.

    Each annotation: [class_id, x_center, y_center, width, height] (normalized 0-1)

    Args:
        image_path: Path to the form image
        fields: List of field dicts with 'region' keys containing bbox coordinates
    """
    img = cv2.imread(image_path)
    ih, iw = img.shape[:2]

    annotations = []
    for field in fields:
        if "region" not in field:
            continue
        region = field["region"]
        x1, y1, x2, y2 = region["x1"], region["y1"], region["x2"], region["y2"]

        # Normalize to 0-1
        x_center = ((x1 + x2) / 2) / iw
        y_center = ((y1 + y2) / 2) / ih
        width = (x2 - x1) / iw
        height = (y2 - y1) / ih

        # Determine class
        field_type = field.get("type", "text")
        if field_type == "checkbox":
            class_id = CLASSES.index("checkbox")
        elif field_type == "date":
            class_id = CLASSES.index("date_field")
        elif field_type == "signature":
            class_id = CLASSES.index("signature")
        elif field_type == "header":
            class_id = CLASSES.index("header")
        else:
            class_id = CLASSES.index("text_field")

        annotations.append({
            "class_id": class_id,
            "x_center": round(x_center, 6),
            "y_center": round(y_center, 6),
            "width": round(width, 6),
            "height": round(height, 6),
        })

    return annotations


def save_yolo_labels(annotations: list[dict], label_path: str):
    """Save annotations in YOLO format (class x_center y_center w h)."""
    with open(label_path, "w") as f:
        for ann in annotations:
            f.write(
                f"{ann['class_id']} {ann['x_center']} {ann['y_center']} "
                f"{ann['width']} {ann['height']}\n"
            )


print("Annotation helpers ready")
print("\nFor proper annotation, use one of:")
print("  - Label Studio (https://labelstud.io/) — free, self-hosted")
print("  - Roboflow (https://roboflow.com/) — free tier, web-based")
print("  - CVAT (https://cvat.ai/) — free, open-source")

In [ ]:
# ============================================================
# Step 5: Interactive Bounding Box Annotator (Notebook-based)
# ============================================================
# This provides a simple visual approach to annotate images
# directly in the notebook for quick prototyping.

def annotate_image_grid(image_path: str, grid_rows: int = 20, grid_cols: int = 4):
    """Show an image with a numbered grid overlay to help identify regions.
    Users can then specify field locations by grid cell numbers."""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(12, 16))
    ax.imshow(img_rgb)

    cell_h = h / grid_rows
    cell_w = w / grid_cols

    for row in range(grid_rows):
        for col in range(grid_cols):
            x = col * cell_w
            y = row * cell_h
            rect = patches.Rectangle(
                (x, y), cell_w, cell_h,
                linewidth=0.5, edgecolor="red", facecolor="none", alpha=0.5
            )
            ax.add_patch(rect)
            cell_num = row * grid_cols + col
            ax.text(
                x + 2, y + cell_h/2, str(cell_num),
                fontsize=6, color="red", alpha=0.7
            )

    ax.set_title(f"Grid Overlay ({grid_rows}x{grid_cols}) — Use numbers for annotation")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    return cell_h, cell_w, h, w


def grid_cells_to_bbox(
    start_cell: int, end_cell: int,
    grid_cols: int, cell_h: float, cell_w: float
) -> tuple[int, int, int, int]:
    """Convert grid cell range to pixel bounding box (x1, y1, x2, y2)."""
    start_row, start_col = divmod(start_cell, grid_cols)
    end_row, end_col = divmod(end_cell, grid_cols)
    x1 = int(start_col * cell_w)
    y1 = int(start_row * cell_h)
    x2 = int((end_col + 1) * cell_w)
    y2 = int((end_row + 1) * cell_h)
    return x1, y1, x2, y2


# Show the grid for the sample image
sample_img = str(IMAGES_DIR / "dti_bnr_sample_01.png")
if Path(sample_img).exists():
    cell_h, cell_w, img_h, img_w = annotate_image_grid(sample_img)
    print(f"\nImage size: {img_w}x{img_h}px")
    print(f"Cell size: {cell_w:.0f}x{cell_h:.0f}px")
    print("\nUse grid cell numbers to define field regions in the next step.")
else:
    print(f"Sample image not found: {sample_img}")

In [ ]:
# ============================================================
# Step 6: Define Annotations for Sample Forms
# ============================================================
# Define bounding boxes manually for the DTI-BNR sample.
# In production, this comes from Label Studio/Roboflow.
#
# Format: field_name, class, x1, y1, x2, y2 (pixel coordinates)
# After verifying with the grid above, enter coordinates here.

# === IMPORTANT: Adjust these coordinates after viewing the grid ===
# These are approximate — update after visual inspection.

DTI_BNR_ANNOTATIONS = [
    # Section A: Business Name
    {"field": "business_name", "class": "text_field", "bbox": None},  # Fill after grid check
    # Section B: Owner Info
    {"field": "last_name", "class": "text_field", "bbox": None},
    {"field": "first_name", "class": "text_field", "bbox": None},
    {"field": "middle_name", "class": "text_field", "bbox": None},
    # Checkboxes
    {"field": "sole_proprietor", "class": "checkbox", "bbox": None},
    {"field": "partnership", "class": "checkbox", "bbox": None},
    {"field": "corporation", "class": "checkbox", "bbox": None},
]

print("\n=== Action Needed ===")
print("1. View the grid in Step 5")
print("2. Identify grid cells for each field")
print("3. Use grid_cells_to_bbox() to get pixel coordinates")
print("4. Fill in the 'bbox' values above")
print("5. Or use Label Studio for proper annotation")
print("\nExample:")
print('  {"field": "business_name", "class": "text_field", "bbox": [100, 200, 500, 230]}')

In [ ]:
# ============================================================
# Step 7: Prepare YOLO Training Dataset
# ============================================================

def prepare_dataset_from_annotations(
    image_paths: list[str],
    all_annotations: list[list[dict]],
    train_split: float = 0.8,
):
    """Split annotated data into train/val and copy to YOLO dataset structure."""
    n = len(image_paths)
    indices = np.random.permutation(n)
    split_idx = int(n * train_split)

    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]

    for split_name, split_indices in [("train", train_indices), ("val", val_indices)]:
        for idx in split_indices:
            img_path = Path(image_paths[idx])
            annotations = all_annotations[idx]

            # Copy image
            dst_img = DATASET_DIR / split_name / "images" / img_path.name
            shutil.copy2(img_path, dst_img)

            # Create YOLO label file
            label_name = img_path.stem + ".txt"
            dst_label = DATASET_DIR / split_name / "labels" / label_name

            img = cv2.imread(str(img_path))
            ih, iw = img.shape[:2]

            with open(dst_label, "w") as f:
                for ann in annotations:
                    if ann.get("bbox") is None:
                        continue
                    x1, y1, x2, y2 = ann["bbox"]
                    class_id = CLASSES.index(ann["class"])
                    # Convert to YOLO normalized format
                    x_center = ((x1 + x2) / 2) / iw
                    y_center = ((y1 + y2) / 2) / ih
                    w = (x2 - x1) / iw
                    h = (y2 - y1) / ih
                    f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        count = len(split_indices)
        print(f"{split_name}: {count} images")


# Example with properly annotated data:
# prepare_dataset_from_annotations(image_paths, all_annotations)

print("Dataset preparation function ready.")
print("\nOnce annotations are complete, run prepare_dataset_from_annotations().")

In [ ]:
# ============================================================
# Step 8: Data Augmentation
# ============================================================
# With limited training data, augmentation is critical.
# YOLOv8 has built-in augmentation, but we can add more:

def augment_form_image(img: np.ndarray) -> list[tuple[np.ndarray, str]]:
    """Generate augmented versions of a form image.
    Returns list of (image, augmentation_name) tuples."""
    augmented = []

    # 1. Slight rotation (simulates misaligned scanning)
    for angle in [-3, -1.5, 1.5, 3]:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        augmented.append((rotated, f"rotate_{angle}"))

    # 2. Brightness variations (different lighting conditions)
    for factor in [0.7, 0.85, 1.15, 1.3]:
        adjusted = cv2.convertScaleAbs(img, alpha=factor, beta=0)
        augmented.append((adjusted, f"brightness_{factor}"))

    # 3. Gaussian noise (simulates low-quality scans)
    for sigma in [10, 25]:
        noise = np.random.normal(0, sigma, img.shape).astype(np.int16)
        noisy = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        augmented.append((noisy, f"noise_{sigma}"))

    # 4. Perspective transform (simulates camera angle)
    h, w = img.shape[:2]
    for offset in [15, 30]:
        pts1 = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
        pts2 = np.float32([
            [offset, offset], [w-offset, 0],
            [0, h-offset], [w-offset, h-offset]
        ])
        M = cv2.getPerspectiveTransform(pts1, pts2)
        warped = cv2.warpPerspective(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        augmented.append((warped, f"perspective_{offset}"))

    return augmented


# Demo augmentation on sample image
sample_img_path = str(IMAGES_DIR / "dti_bnr_sample_01.png")
if Path(sample_img_path).exists():
    img = cv2.imread(sample_img_path)
    augmented = augment_form_image(img)

    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes = axes.flatten()
    for idx, (aug_img, name) in enumerate(augmented[:12]):
        rgb = cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(rgb)
        axes[idx].set_title(name, fontsize=9)
        axes[idx].axis("off")
    plt.suptitle("Data Augmentation Examples", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("augmentation_samples.png", dpi=100, bbox_inches="tight")
    plt.show()

    print(f"Generated {len(augmented)} augmented versions per image")
else:
    print("Sample image not found — skipping augmentation demo")

In [ ]:
# ============================================================
# Step 9: Train YOLOv8
# ============================================================
from ultralytics import YOLO

# Load pretrained model
model = YOLO(f"{MODEL_SIZE}.pt")
print(f"Loaded {MODEL_SIZE} pretrained model")

# Train
# Note: Only run this when you have annotated data in the dataset folder.
# The training will fail if there are no images/labels in train/val folders.

train_images = list((DATASET_DIR / "train" / "images").glob("*.png")) + \
               list((DATASET_DIR / "train" / "images").glob("*.jpg"))

if len(train_images) > 0:
    print(f"Found {len(train_images)} training images. Starting training...")
    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        project=str(MODEL_OUTPUT_DIR),
        name="form_field_detector",
        # Augmentation (built-in)
        augment=True,
        hsv_h=0.015,
        hsv_s=0.4,
        hsv_v=0.4,
        degrees=5.0,
        translate=0.1,
        scale=0.3,
        shear=2.0,
        flipud=0.0,   # Don't flip forms vertically
        fliplr=0.0,   # Don't flip forms horizontally
        mosaic=0.5,
        # Training params
        patience=20,
        lr0=0.01,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        verbose=True,
    )
    print("Training complete!")
else:
    print("No training images found in dataset/train/images/")
    print("Please complete annotation steps first (Steps 5-7).")
    print("\nAlternatively, use Roboflow to annotate and export in YOLO format:")
    print("  1. Upload form images to Roboflow")
    print("  2. Annotate field regions with bounding boxes")
    print("  3. Export as 'YOLOv8' format")
    print("  4. Copy to dataset/train/ and dataset/val/")

In [ ]:
# ============================================================
# Step 10: Evaluate Model
# ============================================================

# Check if a trained model exists
best_model_path = MODEL_OUTPUT_DIR / "form_field_detector" / "weights" / "best.pt"

if best_model_path.exists():
    model = YOLO(str(best_model_path))
    print(f"Loaded best model from {best_model_path}")

    # Validate on validation set
    metrics = model.val(data=str(yaml_path))

    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    print(f"  mAP@0.5:     {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95:{metrics.box.map:.4f}")
    print(f"  Precision:    {metrics.box.mp:.4f}")
    print(f"  Recall:       {metrics.box.mr:.4f}")

    # Per-class results
    print("\nPer-class mAP@0.5:")
    for i, class_name in enumerate(CLASSES):
        if i < len(metrics.box.ap50):
            print(f"  {class_name:15s}: {metrics.box.ap50[i]:.4f}")
else:
    print("No trained model found. Complete training first (Step 9).")
    print(f"Expected model at: {best_model_path}")

In [ ]:
# ============================================================
# Step 11: Visualize Detections
# ============================================================

def visualize_detections(image_path: str, model: "YOLO", conf_threshold: float = 0.25):
    """Run detection and visualize results with bounding boxes."""
    results = model.predict(image_path, conf=conf_threshold, imgsz=IMAGE_SIZE)

    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(14, 18))
    ax.imshow(img_rgb)

    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = CLASSES[cls_id]

            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=colors[cls_id], facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x1, y1-5, f"{class_name} {conf:.2f}",
                fontsize=8, color=colors[cls_id],
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8)
            )

    ax.set_title(f"YOLOv8 Field Detection — {Path(image_path).name}", fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("detection_result.png", dpi=150, bbox_inches="tight")
    plt.show()

    n_detections = sum(len(r.boxes) for r in results)
    print(f"\nDetected {n_detections} field regions")


# Run on sample image if model exists
if best_model_path.exists():
    sample_img = str(IMAGES_DIR / "dti_bnr_sample_01.png")
    if Path(sample_img).exists():
        visualize_detections(sample_img, model)
else:
    print("Train model first to see detection results.")

In [ ]:
# ============================================================
# Step 12: Export Model for Integration
# ============================================================

if best_model_path.exists():
    model = YOLO(str(best_model_path))

    # Export to ONNX for faster inference
    onnx_path = model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True)
    print(f"ONNX model exported to: {onnx_path}")

    # Save model metadata
    metadata = {
        "model": MODEL_SIZE,
        "classes": CLASSES,
        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "onnx_path": str(onnx_path),
        "best_pt_path": str(best_model_path),
        "integration_notes": (
            "Load in ocr_service.py: model = YOLO('best.pt') or ONNX runtime. "
            "Use model.predict(image) to get field bounding boxes. "
            "Replace manual template regions in _extract_regions()."
        ),
    }

    with open(PROJECT_DIR / "model_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nModel metadata saved to {PROJECT_DIR / 'model_metadata.json'}")
    print("\n=== Integration Steps ===")
    print("1. Copy best.pt to backend/app/models/ml/")
    print("2. Update ocr_service.py to load YOLOv8 model")
    print("3. Replace manual region extraction with model predictions")
    print("4. Add confidence threshold filtering (recommended: 0.4)")
else:
    print("No trained model to export. Complete training first.")

---

# Section 4: PaddleOCR Fine-tuning

Fine-tune PaddleOCR recognition model on Philippine handwriting samples.

In [ ]:
# ============================================================
# Step 2: Imports & Configuration
# ============================================================
import json
import os
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
import editdistance
import matplotlib.pyplot as plt

# ---- Configuration ----
SAMPLE_DIR = Path("./sample_data")
IMAGES_DIR = SAMPLE_DIR / "images"
GROUND_TRUTH_PATH = SAMPLE_DIR / "ground_truth.json"

# Training dataset output
TRAIN_DIR = Path("./ocr_finetuning")
CROP_DIR = TRAIN_DIR / "crops"          # Cropped field images
PADDLEOCR_DIR = TRAIN_DIR / "paddle"    # PaddleOCR training format

# PaddleOCR training config
REC_MODEL = "en_PP-OCRv4_rec"  # Base model to fine-tune
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
IMAGE_HEIGHT = 48  # PaddleOCR recognition input height
MAX_TEXT_LEN = 100

# Load ground truth
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

print(f"Loaded ground truth with {len(ground_truth['entries'])} entries")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")

In [ ]:
# ============================================================
# Step 3: Create Cropped Field Images from Full Form
# ============================================================
# Since we're starting with a single form image, we need to
# crop individual field regions to create training samples.
#
# Two approaches:
# A. Use PaddleOCR detection to find text regions automatically
# B. Use manual region definitions from the template

from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=False, show_log=False)


def extract_text_crops(image_path: str) -> list[dict]:
    """Use PaddleOCR detection to extract and crop text regions.

    Returns list of {crop_image, text, confidence, bbox}.
    """
    img = cv2.imread(image_path)
    result = ocr.ocr(img)

    crops = []
    if result and result[0]:
        for idx, line in enumerate(result[0]):
            bbox = line[0]
            text = line[1][0]
            conf = float(line[1][1])

            # Get bounding box as rectangle
            pts = np.array(bbox, dtype=np.int32)
            x_min = max(0, pts[:, 0].min() - 2)
            y_min = max(0, pts[:, 1].min() - 2)
            x_max = min(img.shape[1], pts[:, 0].max() + 2)
            y_max = min(img.shape[0], pts[:, 1].max() + 2)

            crop = img[y_min:y_max, x_min:x_max]
            if crop.size > 0:
                crops.append({
                    "crop": crop,
                    "text": text,
                    "confidence": conf,
                    "bbox": [int(x_min), int(y_min), int(x_max), int(y_max)],
                    "index": idx,
                })

    return crops


# Extract crops from sample image
sample_img = str(IMAGES_DIR / "dti_bnr_sample_01.png")
crops = []  # Initialize in case image is missing

if Path(sample_img).exists():
    crops = extract_text_crops(sample_img)
    print(f"Extracted {len(crops)} text regions from sample image")

    # Show some examples
    n_show = min(12, len(crops))
    fig, axes = plt.subplots(3, 4, figsize=(16, 6))
    axes = axes.flatten()
    for i in range(n_show):
        rgb = cv2.cvtColor(crops[i]["crop"], cv2.COLOR_BGR2RGB)
        axes[i].imshow(rgb)
        axes[i].set_title(f"{crops[i]['text'][:20]} ({crops[i]['confidence']:.2f})", fontsize=7)
        axes[i].axis("off")
    for i in range(n_show, 12):
        axes[i].axis("off")
    plt.suptitle("Detected Text Regions (Auto-cropped)", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Sample image not found: {sample_img}")

In [ ]:
# ============================================================
# Step 4: Match OCR Crops with Ground Truth
# ============================================================
# Pair each auto-detected crop with its correct ground truth label.
# This creates supervised training data: crop_image → correct_text

def match_crops_to_ground_truth(
    crops: list[dict],
    gt_fields: list[dict],
) -> list[dict]:
    """Match OCR crop regions with ground truth verified values.

    Uses text similarity to pair detections with labels.
    Returns list of {crop, ocr_text, verified_text, field_name, confidence}.
    """
    labeled = []

    for field in gt_fields:
        if field["type"] != "text" or not field["verified_value"]:
            continue

        gt_text = field["verified_value"].lower().strip()
        best_match = None
        best_score = 0

        for crop in crops:
            ocr_text = crop["text"].lower().strip()

            # Exact containment
            if gt_text in ocr_text or ocr_text in gt_text:
                score = 0.95
            else:
                # Similarity
                max_len = max(len(gt_text), len(ocr_text))
                if max_len == 0:
                    continue
                dist = editdistance.eval(gt_text, ocr_text)
                score = 1 - (dist / max_len)

            if score > best_score:
                best_score = score
                best_match = crop

        if best_match and best_score >= 0.4:
            labeled.append({
                "crop": best_match["crop"],
                "bbox": best_match["bbox"],
                "ocr_text": best_match["text"],
                "verified_text": field["verified_value"],
                "field_name": field["field_name"],
                "confidence": best_match["confidence"],
                "match_score": round(best_score, 3),
            })

    return labeled


# Match crops with ground truth
entry = ground_truth["entries"][0]
labeled_data = match_crops_to_ground_truth(crops, entry["fields"])

print(f"\nMatched {len(labeled_data)} fields with ground truth")
print(f"{'Field':30s} | {'OCR Text':25s} | {'Verified':25s} | Score")
print("-" * 90)
for item in labeled_data:
    print(
        f"{item['field_name']:30s} | "
        f"{item['ocr_text'][:25]:25s} | "
        f"{item['verified_text'][:25]:25s} | "
        f"{item['match_score']:.3f}"
    )

In [ ]:
# ============================================================
# Step 5: Build PaddleOCR Training Dataset
# ============================================================
# PaddleOCR recognition training format:
# - Images in a folder
# - Label file: image_path\ttext\n

def build_paddle_rec_dataset(
    labeled_data: list[dict],
    output_dir: Path,
    train_split: float = 0.85,
):
    """Create PaddleOCR recognition training dataset files."""
    output_dir.mkdir(parents=True, exist_ok=True)
    img_dir = output_dir / "images"
    img_dir.mkdir(exist_ok=True)

    # Save crops and build label list
    entries = []
    for idx, item in enumerate(labeled_data):
        # Save crop image
        img_name = f"field_{idx:04d}.png"
        img_path = img_dir / img_name
        cv2.imwrite(str(img_path), item["crop"])

        # Resize to fixed height for PaddleOCR
        crop = item["crop"]
        h, w = crop.shape[:2]
        new_h = IMAGE_HEIGHT
        new_w = max(1, int(w * new_h / h))
        resized = cv2.resize(crop, (new_w, new_h))
        cv2.imwrite(str(img_path), resized)

        entries.append({
            "image": f"images/{img_name}",
            "label": item["verified_text"],
        })

    # Shuffle and split
    random.shuffle(entries)
    split_idx = int(len(entries) * train_split)
    train_entries = entries[:split_idx]
    val_entries = entries[split_idx:]

    # Write label files (PaddleOCR format: image_path\tlabel)
    for split_name, split_entries in [("train", train_entries), ("val", val_entries)]:
        label_path = output_dir / f"{split_name}_labels.txt"
        with open(label_path, "w", encoding="utf-8") as f:
            for entry in split_entries:
                f.write(f"{entry['image']}\t{entry['label']}\n")
        print(f"{split_name}: {len(split_entries)} samples → {label_path}")

    # Build character dictionary
    all_chars = set()
    for entry in entries:
        all_chars.update(entry["label"])
    all_chars = sorted(all_chars)

    dict_path = output_dir / "dict.txt"
    with open(dict_path, "w", encoding="utf-8") as f:
        for char in all_chars:
            f.write(f"{char}\n")
    print(f"Character dictionary: {len(all_chars)} chars → {dict_path}")

    return output_dir


# Build dataset
dataset_path = build_paddle_rec_dataset(labeled_data, PADDLEOCR_DIR)
print(f"\nDataset ready at: {dataset_path}")

In [ ]:
# ============================================================
# Step 6: Data Augmentation for Recognition Training
# ============================================================
# With limited data, augmentation is critical.
# We generate variations of each crop to increase training set size.

def augment_text_crop(crop: np.ndarray) -> list[tuple[np.ndarray, str]]:
    """Generate augmented versions of a cropped text field image."""
    augmented = []

    # 1. Slight rotations (-3 to +3 degrees)
    for angle in [-2, -1, 1, 2]:
        h, w = crop.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        rotated = cv2.warpAffine(crop, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        augmented.append((rotated, f"rot{angle}"))

    # 2. Brightness adjustments
    for factor in [0.8, 1.2]:
        adjusted = cv2.convertScaleAbs(crop, alpha=factor, beta=0)
        augmented.append((adjusted, f"bright{factor}"))

    # 3. Gaussian blur (simulates out-of-focus)
    blurred = cv2.GaussianBlur(crop, (3, 3), 0)
    augmented.append((blurred, "blur"))

    # 4. Salt and pepper noise
    noisy = crop.copy()
    prob = 0.01
    rng = np.random.default_rng()
    mask = rng.random(crop.shape[:2])
    noisy[mask < prob] = 0
    noisy[mask > 1 - prob] = 255
    augmented.append((noisy, "noise"))

    # 5. Erosion / Dilation (thicker/thinner strokes)
    kernel = np.ones((2, 2), np.uint8)
    eroded = cv2.erode(crop, kernel, iterations=1)
    augmented.append((eroded, "erode"))
    dilated = cv2.dilate(crop, kernel, iterations=1)
    augmented.append((dilated, "dilate"))

    return augmented


def augment_dataset(paddle_dir: Path, multiplier: int = 5):
    """Augment the training set by creating variations of each image."""
    train_labels_path = paddle_dir / "train_labels.txt"
    img_dir = paddle_dir / "images"

    new_entries = []
    with open(train_labels_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split("\t")
        if len(parts) != 2:
            continue
        img_rel, label = parts
        img_path = paddle_dir / img_rel

        if not img_path.exists():
            continue

        crop = cv2.imread(str(img_path))
        augmented = augment_text_crop(crop)

        # Randomly select augmentations
        selected = random.sample(augmented, min(multiplier, len(augmented)))

        for aug_img, aug_name in selected:
            stem = img_path.stem
            aug_filename = f"{stem}_{aug_name}.png"
            aug_path = img_dir / aug_filename
            cv2.imwrite(str(aug_path), aug_img)
            new_entries.append(f"images/{aug_filename}\t{label}")

    # Append augmented entries to training labels
    with open(train_labels_path, "a", encoding="utf-8") as f:
        for entry in new_entries:
            f.write(entry + "\n")

    total = len(lines) + len(new_entries)
    print(f"Augmented training set: {len(lines)} original + {len(new_entries)} augmented = {total} total")


# Run augmentation
augment_dataset(PADDLEOCR_DIR, multiplier=5)

# Show augmentation examples
if len(labeled_data) > 0:
    sample_crop = labeled_data[0]["crop"]
    aug_samples = augment_text_crop(sample_crop)

    fig, axes = plt.subplots(2, 5, figsize=(16, 4))
    axes = axes.flatten()
    axes[0].imshow(cv2.cvtColor(sample_crop, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original", fontsize=9)
    axes[0].axis("off")
    for i, (aug, name) in enumerate(aug_samples[:9]):
        axes[i+1].imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
        axes[i+1].set_title(name, fontsize=9)
        axes[i+1].axis("off")
    plt.suptitle(f"Augmentation Examples: '{labeled_data[0]['verified_text'][:30]}'", fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Step 7: PaddleOCR Training Configuration
# ============================================================
# Create the YAML config file for PaddleOCR recognition fine-tuning.

paddle_config = {
    "Global": {
        "use_gpu": True,
        "epoch_num": EPOCHS,
        "log_smooth_window": 20,
        "print_batch_step": 10,
        "save_model_dir": str(TRAIN_DIR / "output" / "rec_finetune"),
        "save_epoch_step": 10,
        "eval_batch_step": [0, 50],
        "cal_metric_during_train": True,
        "pretrained_model": None,  # Will be set to downloaded model path
        "checkpoints": None,
        "save_inference_dir": str(TRAIN_DIR / "output" / "inference"),
        "use_visualdl": False,
        "infer_img": None,
        "character_dict_path": str(PADDLEOCR_DIR / "dict.txt"),
        "max_text_length": MAX_TEXT_LEN,
        "infer_mode": False,
        "use_space_char": True,
    },
    "Optimizer": {
        "name": "Adam",
        "beta1": 0.9,
        "beta2": 0.999,
        "lr": {
            "name": "Cosine",
            "learning_rate": LEARNING_RATE,
            "warmup_epoch": 5,
        },
        "regularizer": {
            "name": "L2",
            "factor": 0.00001,
        },
    },
    "Architecture": {
        "model_type": "rec",
        "algorithm": "SVTR_LCNet",
        "Transform": None,
        "Backbone": {
            "name": "MobileNetV1Enhance",
            "scale": 0.5,
            "last_conv_stride": [1, 2],
            "last_pool_type": "avg",
        },
        "Head": {
            "name": "MultiHead",
            "head_list": [
                {
                    "CTCHead": {
                        "Neck": {"name": "svtr", "dims": 64, "depth": 2, "hidden_dims": 120, "use_guide": True},
                        "Head": {"fc_decay": 0.00001},
                    }
                },
                {
                    "SARHead": {
                        "enc_dim": 512,
                        "max_text_length": MAX_TEXT_LEN,
                    }
                },
            ],
        },
    },
    "Loss": {
        "name": "MultiLoss",
        "loss_config_list": [
            {"CTCLoss": None},
            {"SARLoss": None},
        ],
    },
    "PostProcess": {
        "name": "CTCLabelDecode",
    },
    "Metric": {
        "name": "RecMetric",
        "main_indicator": "acc",
        "ignore_space": False,
    },
    "Train": {
        "dataset": {
            "name": "SimpleDataSet",
            "data_dir": str(PADDLEOCR_DIR),
            "label_file_list": [str(PADDLEOCR_DIR / "train_labels.txt")],
            "transforms": [
                {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                {"RecConAug": {"prob": 0.5, "max_text_length": MAX_TEXT_LEN}},
                {"RecAug": {}},
                {"MultiLabelEncode": {}},
                {"RecResizeImg": {"image_shape": [3, IMAGE_HEIGHT, 320]}},
                {"KeepKeys": {"keep_keys": ["image", "label_ctc", "label_sar", "length", "valid_ratio"]}},
            ],
        },
        "loader": {
            "shuffle": True,
            "batch_size_per_card": BATCH_SIZE,
            "drop_last": True,
            "num_workers": 4,
        },
    },
    "Eval": {
        "dataset": {
            "name": "SimpleDataSet",
            "data_dir": str(PADDLEOCR_DIR),
            "label_file_list": [str(PADDLEOCR_DIR / "val_labels.txt")],
            "transforms": [
                {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                {"MultiLabelEncode": {}},
                {"RecResizeImg": {"image_shape": [3, IMAGE_HEIGHT, 320]}},
                {"KeepKeys": {"keep_keys": ["image", "label_ctc", "label_sar", "length", "valid_ratio"]}},
            ],
        },
        "loader": {
            "shuffle": False,
            "drop_last": False,
            "batch_size_per_card": BATCH_SIZE,
            "num_workers": 2,
        },
    },
}

# Save config
import yaml

config_path = TRAIN_DIR / "rec_finetune.yml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(paddle_config, f, default_flow_style=False)

print(f"Training config saved to: {config_path}")
print(f"\nTo train, run:")
print(f"  python PaddleOCR/tools/train.py -c {config_path}")

In [ ]:
# ============================================================
# Step 8: Download Pretrained Model & Start Training
# ============================================================
# Uncomment the following for Google Colab execution:

# # Download PP-OCRv4 English recognition pretrained model
# !wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar
# !tar -xf en_PP-OCRv4_rec_train.tar

# # Update config with pretrained model path
# import yaml
# with open(str(config_path)) as f:
#     config = yaml.safe_load(f)
# config["Global"]["pretrained_model"] = "./en_PP-OCRv4_rec_train/best_accuracy"
# with open(str(config_path), "w") as f:
#     yaml.dump(config, f, default_flow_style=False)

# # Start training
# !python PaddleOCR/tools/train.py -c {str(config_path)}

print("=== Training Instructions ===")
print("")
print("On Google Colab:")
print("1. Uncomment and run the cell above")
print("2. Training will use the T4 GPU")
print(f"3. Model checkpoints saved to: {TRAIN_DIR / 'output' / 'rec_finetune'}")
print("")
print("Locally (with GPU):")
print(f"  python PaddleOCR/tools/train.py -c {config_path}")
print("")
print("Resume from checkpoint:")
print(f"  python PaddleOCR/tools/train.py -c {config_path} -o Global.checkpoints=./output/rec_finetune/latest")

In [ ]:
# ============================================================
# Step 9: Evaluate Fine-tuned Model
# ============================================================

def evaluate_finetuned_model(
    model_dir: str,
    test_images: list[str],
    test_labels: list[str],
) -> dict:
    """Evaluate the fine-tuned PaddleOCR model on test data."""
    finetuned_ocr = PaddleOCR(
        rec_model_dir=model_dir,
        use_angle_cls=True,
        lang="en",
        use_gpu=False,
        show_log=False,
    )

    total_cer = 0
    total_wer = 0
    exact_matches = 0
    results = []

    for img_path, gt_text in zip(test_images, test_labels):
        img = cv2.imread(img_path)
        result = finetuned_ocr.ocr(img)

        ocr_text = ""
        if result and result[0]:
            ocr_text = result[0][0][1][0]  # First line text

        cer = editdistance.eval(ocr_text, gt_text) / max(len(gt_text), 1)
        wer_val = editdistance.eval(ocr_text.split(), gt_text.split()) / max(len(gt_text.split()), 1)

        total_cer += cer
        total_wer += wer_val
        if ocr_text.strip().lower() == gt_text.strip().lower():
            exact_matches += 1

        results.append({
            "image": img_path,
            "gt": gt_text,
            "pred": ocr_text,
            "cer": round(cer, 4),
        })

    n = len(test_images)
    return {
        "n_samples": n,
        "avg_cer": round(total_cer / n * 100, 2) if n else 0,
        "avg_wer": round(total_wer / n * 100, 2) if n else 0,
        "exact_match_rate": round(exact_matches / n * 100, 2) if n else 0,
        "details": results,
    }


# Check for trained model
model_output = TRAIN_DIR / "output" / "rec_finetune"
if model_output.exists():
    print(f"Found trained model at: {model_output}")
    # Load val labels for evaluation
    val_labels_path = PADDLEOCR_DIR / "val_labels.txt"
    if val_labels_path.exists():
        test_images = []
        test_labels = []
        with open(val_labels_path) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) == 2:
                    test_images.append(str(PADDLEOCR_DIR / parts[0]))
                    test_labels.append(parts[1])

        eval_results = evaluate_finetuned_model(
            str(model_output / "best_accuracy"),
            test_images,
            test_labels,
        )

        print(f"\n{'='*60}")
        print("FINE-TUNED MODEL EVALUATION")
        print(f"{'='*60}")
        print(f"  Samples:          {eval_results['n_samples']}")
        print(f"  Avg CER:          {eval_results['avg_cer']}%")
        print(f"  Avg WER:          {eval_results['avg_wer']}%")
        print(f"  Exact Match Rate: {eval_results['exact_match_rate']}%")
else:
    print("No trained model found. Complete training first (Step 8).")

In [ ]:
# ============================================================
# Step 10: Compare Baseline vs Fine-tuned
# ============================================================

def compare_models(image_path: str, gt_fields: list[dict]):
    """Compare baseline PaddleOCR vs fine-tuned model on the same image."""
    # Baseline
    baseline_ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=False, show_log=False)

    img = cv2.imread(image_path)
    baseline_result = baseline_ocr.ocr(img)
    baseline_lines = []
    if baseline_result and baseline_result[0]:
        baseline_lines = [{"text": l[1][0], "conf": l[1][1]} for l in baseline_result[0]]

    # Fine-tuned (if available)
    ft_model_path = TRAIN_DIR / "output" / "rec_finetune" / "best_accuracy"
    ft_lines = []
    if ft_model_path.exists():
        ft_ocr = PaddleOCR(
            rec_model_dir=str(ft_model_path),
            use_angle_cls=True, lang="en", use_gpu=False, show_log=False,
        )
        ft_result = ft_ocr.ocr(img)
        if ft_result and ft_result[0]:
            ft_lines = [{"text": l[1][0], "conf": l[1][1]} for l in ft_result[0]]

    # Compare field matching
    text_fields = [f for f in gt_fields if f["type"] == "text" and f["verified_value"]]

    print(f"{'Field':30s} | {'Baseline':25s} | {'Fine-tuned':25s} | {'Ground Truth':25s}")
    print("-" * 110)

    baseline_matches = 0
    ft_matches = 0

    for field in text_fields:
        gt_val = field["verified_value"]

        # Find in baseline
        b_match = "—"
        for line in baseline_lines:
            if gt_val.lower() in line["text"].lower() or line["text"].lower() in gt_val.lower():
                b_match = line["text"][:25]
                baseline_matches += 1
                break

        # Find in fine-tuned
        ft_match = "—"
        for line in ft_lines:
            if gt_val.lower() in line["text"].lower() or line["text"].lower() in gt_val.lower():
                ft_match = line["text"][:25]
                ft_matches += 1
                break

        print(f"{field['field_name']:30s} | {b_match:25s} | {ft_match:25s} | {gt_val[:25]:25s}")

    print(f"\nBaseline matches:   {baseline_matches}/{len(text_fields)}")
    if ft_lines:
        print(f"Fine-tuned matches: {ft_matches}/{len(text_fields)}")
        improvement = ft_matches - baseline_matches
        print(f"Improvement: {'+' if improvement >= 0 else ''}{improvement} fields")
    else:
        print("Fine-tuned model not available for comparison.")


# Run comparison
sample_img = str(IMAGES_DIR / "dti_bnr_sample_01.png")
if Path(sample_img).exists():
    entry = ground_truth["entries"][0]
    compare_models(sample_img, entry["fields"])

In [ ]:
# ============================================================
# Step 11: Export Model for Deployment
# ============================================================

# Uncomment for Colab:
# !python PaddleOCR/tools/export_model.py \
#     -c {str(config_path)} \
#     -o Global.pretrained_model={str(TRAIN_DIR / 'output' / 'rec_finetune' / 'best_accuracy')} \
#     Global.save_inference_dir={str(TRAIN_DIR / 'output' / 'inference')}

inference_dir = TRAIN_DIR / "output" / "inference"
if inference_dir.exists():
    print(f"Inference model exported to: {inference_dir}")
    print(f"\nFiles:")
    for f in inference_dir.iterdir():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}: {size_kb:.1f} KB")

    print(f"\n=== Integration Steps ===")
    print(f"1. Copy inference model to backend/app/models/ml/paddleocr_rec/")
    print(f"2. Update ocr_service.py _get_paddle_ocr() to load fine-tuned model:")
    print(f"   ocr = PaddleOCR(rec_model_dir='models/ml/paddleocr_rec/', ...)")
    print(f"3. Test on a new form image")
    print(f"4. Compare accuracy before/after using notebook 07")
else:
    print("Export the model first by running the export command above.")
    print(f"Expected output: {inference_dir}")

---

# Section 5: Field Mapping (LayoutLMv3)

Train a LayoutLMv3 model to classify OCR tokens into form field categories using text + layout features.

In [ ]:
# ============================================================
# Step 2: Imports & Configuration
# ============================================================
import json
import os
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
from PIL import Image
import torch
import editdistance
import matplotlib.pyplot as plt

# ---- Configuration ----
SAMPLE_DIR = Path("./sample_data")
IMAGES_DIR = SAMPLE_DIR / "images"
GROUND_TRUTH_PATH = SAMPLE_DIR / "ground_truth.json"
MODEL_DIR = Path("./field_mapping")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Training parameters
MODEL_NAME = "microsoft/layoutlmv3-base"
EPOCHS = 30
BATCH_SIZE = 4       # Small batch for Colab T4
LEARNING_RATE = 5e-5
MAX_SEQ_LENGTH = 512

# Load ground truth
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

print(f"Ground truth entries: {len(ground_truth['entries'])}")
print(f"Model: {MODEL_NAME}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ============================================================
# Step 3: Define Label Schema
# ============================================================
# BIO tagging scheme: B-field (beginning), I-field (inside), O (other/non-field)

# Get all unique field names from ground truth
all_field_names = set()
for entry in ground_truth["entries"]:
    for field in entry["fields"]:
        all_field_names.add(field["field_name"])

all_field_names = sorted(all_field_names)

# Create BIO labels
labels = ["O"]  # Other/non-field text
for field in all_field_names:
    labels.append(f"B-{field}")  # Beginning of field
    labels.append(f"I-{field}")  # Inside (continuation) of field

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

NUM_LABELS = len(labels)
print(f"Total field names: {len(all_field_names)}")
print(f"Total BIO labels: {NUM_LABELS}")
print(f"\nField names: {all_field_names[:10]}...")
print(f"\nSample labels: {labels[:10]}...")

In [ ]:
# ============================================================
# Step 4: Run PaddleOCR to Get Text + Bounding Boxes
# ============================================================
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=False, show_log=False)


def get_ocr_with_boxes(image_path: str) -> list[dict]:
    """Run OCR and return text tokens with bounding boxes."""
    img = cv2.imread(image_path)
    ih, iw = img.shape[:2]
    result = ocr.ocr(img)

    tokens = []
    if result and result[0]:
        for line in result[0]:
            bbox = line[0]
            text = line[1][0]
            conf = float(line[1][1])

            # Convert polygon to rectangle
            pts = np.array(bbox)
            x_min = int(pts[:, 0].min())
            y_min = int(pts[:, 1].min())
            x_max = int(pts[:, 0].max())
            y_max = int(pts[:, 1].max())

            # Normalize bbox to 0-1000 (LayoutLMv3 convention)
            norm_bbox = [
                int(x_min / iw * 1000),
                int(y_min / ih * 1000),
                int(x_max / iw * 1000),
                int(y_max / ih * 1000),
            ]

            # Split text into words (tokens)
            words = text.split()
            n_words = len(words)
            word_width = (x_max - x_min) / max(n_words, 1)

            for w_idx, word in enumerate(words):
                # Approximate per-word bbox
                w_x_min = x_min + int(w_idx * word_width)
                w_x_max = x_min + int((w_idx + 1) * word_width)
                word_bbox = [
                    int(w_x_min / iw * 1000),
                    int(y_min / ih * 1000),
                    int(w_x_max / iw * 1000),
                    int(y_max / ih * 1000),
                ]
                tokens.append({
                    "text": word,
                    "bbox": word_bbox,
                    "line_bbox": norm_bbox,
                    "confidence": conf,
                })

    return tokens


# Run on sample image
sample_img = str(IMAGES_DIR / "dti_bnr_sample_01.png")
ocr_tokens = []  # Initialize in case image is missing
ocr_boxes = []

if Path(sample_img).exists():
    ocr_tokens = get_ocr_with_boxes(sample_img)
    print(f"Extracted {len(ocr_tokens)} tokens")
    print("\nFirst 10 tokens:")
    for t in ocr_tokens[:10]:
        print(f"  '{t['text']:20s}' bbox={t['bbox']} conf={t['confidence']:.2f}")

In [ ]:
# ============================================================
# Step 5: Label OCR Tokens with Ground Truth
# ============================================================
# Match each OCR token to a ground truth field using text similarity.

def label_tokens(
    ocr_tokens: list[dict],
    gt_fields: list[dict],
) -> list[dict]:
    """Assign BIO labels to OCR tokens based on ground truth.

    For each ground truth field value, find matching OCR tokens
    and label them with B-field/I-field tags.
    Non-matching tokens get 'O' (Other).
    """
    # Initialize all tokens as 'O'
    labeled = []
    for token in ocr_tokens:
        labeled.append({**token, "label": "O", "field_name": None})

    # For each ground truth field, find matching tokens
    used_indices = set()

    for field in gt_fields:
        if not field.get("verified_value"):
            continue

        gt_words = field["verified_value"].lower().split()
        if not gt_words:
            continue

        field_name = field["field_name"]

        # Sliding window search over tokens
        best_start = -1
        best_score = 0

        for i in range(len(labeled)):
            if i in used_indices:
                continue

            # Try matching gt_words starting at position i
            window = []
            for j in range(i, min(i + len(gt_words) + 2, len(labeled))):
                if j in used_indices:
                    break
                window.append(labeled[j]["text"].lower())

            if not window:
                continue

            # Score: how well do these tokens match the GT words?
            window_text = " ".join(window)
            gt_text = " ".join(gt_words)

            max_len = max(len(window_text), len(gt_text))
            if max_len == 0:
                continue
            dist = editdistance.eval(window_text, gt_text)
            score = 1 - (dist / max_len)

            if score > best_score:
                best_score = score
                best_start = i

        # Label matched tokens
        if best_score >= 0.4 and best_start >= 0:
            n_tokens = min(len(gt_words) + 1, len(labeled) - best_start)
            for j in range(n_tokens):
                idx = best_start + j
                if idx in used_indices:
                    break
                if j == 0:
                    labeled[idx]["label"] = f"B-{field_name}"
                else:
                    labeled[idx]["label"] = f"I-{field_name}"
                labeled[idx]["field_name"] = field_name
                used_indices.add(idx)

    return labeled


# Label tokens
entry = ground_truth["entries"][0]
labeled_tokens = label_tokens(ocr_tokens, entry["fields"])

# Summary
label_counts = Counter(t["label"] for t in labeled_tokens)
print(f"Total tokens: {len(labeled_tokens)}")
print(f"Labeled (non-O): {len(labeled_tokens) - label_counts.get('O', 0)}")
print(f"\nLabel distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:30s}: {count}")

In [ ]:
# ============================================================
# Step 6: Prepare LayoutLMv3 Training Data
# ============================================================
from transformers import LayoutLMv3Processor, LayoutLMv3TokenizerFast

processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)


def prepare_layoutlm_input(
    labeled_tokens: list[dict],
    image_path: str,
) -> dict:
    """Convert labeled OCR tokens to LayoutLMv3 input format."""
    words = [t["text"] for t in labeled_tokens]
    boxes = [t["bbox"] for t in labeled_tokens]
    word_labels = [label2id.get(t["label"], 0) for t in labeled_tokens]

    image = Image.open(image_path).convert("RGB")

    # Encode with processor
    encoding = processor(
        image,
        words,
        boxes=boxes,
        word_labels=word_labels,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "bbox": encoding["bbox"].squeeze(0),
        "pixel_values": encoding["pixel_values"].squeeze(0),
        "labels": encoding["labels"].squeeze(0),
    }


# Prepare sample input
sample_input = prepare_layoutlm_input(labeled_tokens, sample_img)
print(f"Input shapes:")
for key, tensor in sample_input.items():
    print(f"  {key}: {tensor.shape}")

In [ ]:
# ============================================================
# Step 7: Create PyTorch Dataset
# ============================================================
from torch.utils.data import Dataset, DataLoader


class FormFieldDataset(Dataset):
    """Dataset for LayoutLMv3 form field classification."""

    def __init__(self, entries: list[dict], images_dir: Path, processor):
        self.entries = entries
        self.images_dir = images_dir
        self.processor = processor
        self.samples = []
        self._prepare()

    def _prepare(self):
        """Pre-process all entries."""
        for entry in self.entries:
            img_path = str(self.images_dir.parent / entry["image_file"])
            if not Path(img_path).exists():
                continue

            # Get OCR tokens
            tokens = get_ocr_with_boxes(img_path)

            # Label tokens
            labeled = label_tokens(tokens, entry["fields"])

            self.samples.append({
                "image_path": img_path,
                "labeled_tokens": labeled,
            })

        print(f"Prepared {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return prepare_layoutlm_input(
            sample["labeled_tokens"],
            sample["image_path"],
        )


# Create dataset
# For MVP with limited data, we use the same entries for train/val
# In production, split across different form images
dataset = FormFieldDataset(
    ground_truth["entries"],
    IMAGES_DIR,
    processor,
)

print(f"Dataset size: {len(dataset)}")
if len(dataset) > 0:
    sample = dataset[0]
    print(f"Sample input_ids shape: {sample['input_ids'].shape}")
    print(f"Sample bbox shape: {sample['bbox'].shape}")

In [ ]:
# ============================================================
# Step 8: Train LayoutLMv3
# ============================================================
from transformers import (
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
)

# Load model
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,  # Evaluate every 50 steps
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",  # Set to "wandb" for experiment tracking
    remove_unused_columns=False,
)

print(f"Training config: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

In [ ]:
# ============================================================
# Step 9: Define Evaluation Metrics
# ============================================================
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score


def compute_metrics(pred):
    """Compute seqeval metrics for token classification."""
    predictions = pred.predictions
    labels = pred.label_ids

    # Convert logits to label indices
    pred_ids = np.argmax(predictions, axis=2)

    # Build true/pred label sequences (ignoring special tokens with label -100)
    true_labels = []
    pred_labels = []

    for i in range(labels.shape[0]):
        true_seq = []
        pred_seq = []
        for j in range(labels.shape[1]):
            if labels[i][j] != -100:
                true_seq.append(id2label[labels[i][j]])
                pred_seq.append(id2label[pred_ids[i][j]])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq)

    f1 = f1_score(true_labels, pred_labels)
    precision = precision_score(true_labels, pred_labels)
    recall = recall_score(true_labels, pred_labels)

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


print("Evaluation metrics ready (seqeval F1, precision, recall)")

In [ ]:
# ============================================================
# Step 10: Run Training
# ============================================================

if len(dataset) > 0:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        eval_dataset=dataset,  # Same for MVP; use separate val set in production
        compute_metrics=compute_metrics,
    )

    print("Starting training...")
    train_result = trainer.train()

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"  Training loss: {train_result.training_loss:.4f}")

    # Evaluate
    eval_results = trainer.evaluate()
    print(f"  Eval F1:       {eval_results.get('eval_f1', 'N/A')}")
    print(f"  Eval Precision:{eval_results.get('eval_precision', 'N/A')}")
    print(f"  Eval Recall:   {eval_results.get('eval_recall', 'N/A')}")

    # Save best model
    best_model_dir = MODEL_DIR / "best_model"
    trainer.save_model(str(best_model_dir))
    processor.save_pretrained(str(best_model_dir))
    print(f"\nModel saved to: {best_model_dir}")
else:
    print("No training data available. Ensure ground truth + images are present.")

In [ ]:
# ============================================================
# Step 12: Inference — Predict Fields on New Form
# ============================================================

def predict_fields(
    image_path: str,
    model,
    processor,
    ocr,
    id2label: dict,
    conf_threshold: float = 0.5,
) -> list[dict]:
    """Run OCR + LayoutLMv3 inference on a new form image."""
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    # Run PaddleOCR
    result = ocr.ocr(image_path)
    if not result or not result[0]:
        return []

    tokens = []
    boxes = []
    for line in result[0]:
        box_pts, (text, conf) = line
        if conf < conf_threshold:
            continue
        x_coords = [p[0] for p in box_pts]
        y_coords = [p[1] for p in box_pts]
        norm_box = [
            int(min(x_coords) / w * 1000),
            int(min(y_coords) / h * 1000),
            int(max(x_coords) / w * 1000),
            int(max(y_coords) / h * 1000),
        ]
        tokens.append(text)
        boxes.append(norm_box)

    if not tokens:
        return []

    # Encode with processor
    encoding = processor(
        image,
        tokens,
        boxes=boxes,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=False,
    )

    # Get word_ids for proper subword → word alignment
    word_ids = encoding.word_ids(batch_index=0)

    # Move tensors to model device
    device = next(model.parameters()).device
    encoding = {k: v.to(device) if hasattr(v, 'to') else v for k, v in encoding.items()}

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**{k: v for k, v in encoding.items() if k != 'offset_mapping'})

    predictions = outputs.logits.argmax(-1).squeeze().tolist()

    # Align predictions back to words using word_ids
    # Only take the prediction for the FIRST subword token of each word
    word_predictions = {}
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:  # Special tokens [CLS], [SEP], [PAD]
            continue
        if word_id not in word_predictions:  # First subword of the word
            word_predictions[word_id] = predictions[token_idx]

    # Build field results
    fields = {}
    for word_id, pred_id in word_predictions.items():
        if word_id >= len(tokens):
            continue
        label = id2label.get(pred_id, "O")
        if label == "O":
            continue

        # Parse BIO label: B-field_name or I-field_name
        bio_tag = label[0]  # B or I
        field_name = label[2:]  # field name after B- or I-

        if bio_tag == "B":
            fields[field_name] = tokens[word_id]
        elif bio_tag == "I" and field_name in fields:
            fields[field_name] += " " + tokens[word_id]

    return [{"field": k, "value": v} for k, v in fields.items()]


# Test inference
sample_img = str(SAMPLE_DIR / ground_truth["entries"][0]["image_file"])
if Path(sample_img).exists():
    predicted = predict_fields(sample_img, model, processor, ocr, id2label)
    print(f"Predicted {len(predicted)} fields:")
    for f in predicted:
        print(f"  {f['field']:30s} → {f['value']}")
else:
    print(f"Image not found: {sample_img}")
    predicted = []

In [ ]:
# ============================================================
# Step 12: Compare with LLM-based Mapping
# ============================================================

def compare_with_ground_truth(predictions: list[dict], gt_fields: list[dict]) -> dict:
    """Compare LayoutLMv3 predictions against ground truth."""
    gt_dict = {f["field_name"]: f["verified_value"] for f in gt_fields if f.get("verified_value")}
    pred_dict = {p["field"]: p["value"] for p in predictions}

    correct = 0
    partial = 0
    missing = 0
    extra = 0
    details = []

    for field_name, gt_value in gt_dict.items():
        pred_value = pred_dict.get(field_name, "")
        if pred_value.lower().strip() == gt_value.lower().strip():
            correct += 1
            status = "EXACT"
        elif pred_value and (
            gt_value.lower() in pred_value.lower()
            or pred_value.lower() in gt_value.lower()
        ):
            partial += 1
            status = "PARTIAL"
        elif not pred_value:
            missing += 1
            status = "MISSING"
        else:
            status = "WRONG"

        details.append({
            "field": field_name,
            "gt": gt_value,
            "pred": pred_value,
            "status": status,
        })

    # Fields predicted but not in GT
    for field_name in pred_dict:
        if field_name not in gt_dict:
            extra += 1

    n_gt = len(gt_dict)
    return {
        "exact_match": correct,
        "partial_match": partial,
        "missing": missing,
        "extra_predictions": extra,
        "total_gt_fields": n_gt,
        "accuracy": round((correct + partial) / max(n_gt, 1) * 100, 2),
        "details": details,
    }


if best_model_dir.exists():
    entry = ground_truth["entries"][0]
    predictions = predict_fields(sample_img, model, processor, ocr, id2label)
    comparison = compare_with_ground_truth(predictions, entry["fields"])

    print(f"{'='*70}")
    print("LayoutLMv3 vs Ground Truth")
    print(f"{'='*70}")
    print(f"  Exact matches:  {comparison['exact_match']}/{comparison['total_gt_fields']}")
    print(f"  Partial matches:{comparison['partial_match']}/{comparison['total_gt_fields']}")
    print(f"  Missing:        {comparison['missing']}/{comparison['total_gt_fields']}")
    print(f"  Accuracy:       {comparison['accuracy']}%")

    print(f"\n{'Field':30s} | {'Status':8s} | {'Predicted':25s} | {'Ground Truth':25s}")
    print("-" * 95)
    for d in comparison["details"]:
        print(f"{d['field']:30s} | {d['status']:8s} | {d['pred'][:25]:25s} | {d['gt'][:25]:25s}")
else:
    print("Train the model first to run comparison.")

In [ ]:
# ============================================================
# Step 13: Export for Integration
# ============================================================

if best_model_dir.exists():
    # Save label mapping
    mapping = {
        "model_name": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "label2id": label2id,
        "id2label": id2label,
        "field_names": list(all_field_names),
        "max_seq_length": MAX_SEQ_LENGTH,
        "integration_notes": (
            "Replace _map_fields_with_ai() in ocr_service.py with LayoutLMv3 prediction. "
            "Load model with LayoutLMv3ForTokenClassification.from_pretrained(model_dir). "
            "Use predict_fields() function from this notebook. "
            "Eliminates Groq API dependency for field mapping."
        ),
    }

    with open(MODEL_DIR / "model_config.json", "w") as f:
        json.dump(mapping, f, indent=2)

    print(f"Model config saved to: {MODEL_DIR / 'model_config.json'}")
    print(f"\n=== Integration Steps ===")
    print(f"1. Copy {best_model_dir}/ to backend/app/models/ml/layoutlmv3/")
    print(f"2. In ocr_service.py, replace _map_fields_with_ai() body:")
    print(f"   - Load LayoutLMv3ForTokenClassification")
    print(f"   - Use predict_fields() logic")
    print(f"   - Return mapped fields dict")
    print(f"3. Remove Groq API dependency for field mapping")
    print(f"4. Test on new form images")
else:
    print("Train model first to export.")

---

# Section 6: Post-OCR Text Correction

Build a correction pipeline (rule-based + T5) that fixes common OCR errors in Filipino/English text.

In [ ]:
# ============================================================
# Step 2: Imports & Configuration
# ============================================================
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import editdistance
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Configuration ----
SAMPLE_DIR = Path("./sample_data")
GROUND_TRUTH_PATH = SAMPLE_DIR / "ground_truth.json"
OUTPUT_DIR = Path("./post_correction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load ground truth
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

print(f"Loaded ground truth with {len(ground_truth['entries'])} entries")

In [ ]:
# ============================================================
# Step 3: Build Error Analysis from OCR vs Ground Truth
# ============================================================
# Analyze common OCR errors to build targeted correction rules.
#
# NOTE: ground_truth.json may not have 'ocr_value' (only 'verified_value').
# When running on real data from the API (NB01 export), both fields exist.
# For local testing, we generate synthetic OCR errors to analyze.

def analyze_ocr_errors(ocr_text: str, gt_text: str) -> dict:
    """Analyze character-level errors between OCR output and ground truth."""
    errors = {
        "substitutions": [],
        "insertions": [],
        "deletions": [],
        "case_errors": 0,
        "space_errors": 0,
    }

    if ocr_text.lower() == gt_text.lower() and ocr_text != gt_text:
        errors["case_errors"] = sum(
            1 for a, b in zip(ocr_text, gt_text) if a != b
        )
        return errors

    if ocr_text.replace(" ", "") == gt_text.replace(" ", ""):
        errors["space_errors"] = 1
        return errors

    ocr_chars = list(ocr_text)
    gt_chars = list(gt_text)

    for i in range(min(len(ocr_chars), len(gt_chars))):
        if ocr_chars[i] != gt_chars[i]:
            errors["substitutions"].append((ocr_chars[i], gt_chars[i]))

    if len(ocr_chars) > len(gt_chars):
        errors["insertions"] = ocr_chars[len(gt_chars):]
    elif len(gt_chars) > len(ocr_chars):
        errors["deletions"] = gt_chars[len(ocr_chars):]

    return errors


# Collect OCR error patterns
# If ground truth has ocr_value → use real pairs
# Otherwise, note that error analysis requires API-exported data
all_substitutions = Counter()
error_categories = Counter()
real_pairs_found = 0

for entry in ground_truth["entries"]:
    for field in entry["fields"]:
        ocr_val = field.get("ocr_value", "")
        gt_val = field.get("verified_value", "")

        if not ocr_val or not gt_val or ocr_val == gt_val:
            continue

        real_pairs_found += 1
        errors = analyze_ocr_errors(ocr_val, gt_val)

        for sub in errors["substitutions"]:
            all_substitutions[sub] += 1

        if errors["case_errors"]:
            error_categories["case_only"] += 1
        elif errors["space_errors"]:
            error_categories["spacing_only"] += 1
        elif errors["substitutions"]:
            error_categories["character_substitution"] += 1
        else:
            error_categories["other"] += 1

if real_pairs_found == 0:
    print("NOTE: ground_truth.json has no 'ocr_value' fields.")
    print("Error analysis requires data exported from the API (notebook 01).")
    print("Skipping to rule-based corrections and synthetic data generation.")
else:
    print(f"Analyzed {real_pairs_found} OCR error pairs.")
    print(f"\nError Categories:")
    for cat, count in error_categories.most_common():
        print(f"  {cat}: {count}")
    print(f"\nTop Character Substitutions (OCR → Correct):")
    for (ocr_char, gt_char), count in all_substitutions.most_common(20):
        print(f"  '{ocr_char}' → '{gt_char}': {count}x")

In [ ]:
# ============================================================
# Step 4: Rule-Based Correction Engine
# ============================================================

# Common OCR confusion pairs for handwritten text
OCR_CONFUSION_MAP = {
    # Digits ↔ Letters
    "0": "O", "O": "0",
    # Digits ↔ Letters (multi-char confusions handled separately)
    "0": "O", "O": "0",
    "5": "S", "S": "5",
    "8": "B", "B": "8",
    "6": "G", "G": "6",
    "5": "S", "S": "5",
    "8": "B", "B": "8",
    "6": "G", "G": "6",
    # Similar characters
    "rn": "m",
    "cl": "d",
    "vv": "w",
    "ii": "u",
}

# Philippine-specific patterns
PH_CORRECTIONS = {
    # Common OCR errors for Filipino words
    "brgy": "Brgy.",
    "brgy.": "Brgy.",
    "BRGY": "Brgy.",
    "barangay": "Barangay",
    "mun.": "Mun.",
    "prov.": "Prov.",
    "st.": "St.",
    "ave.": "Ave.",
    "blvd.": "Blvd.",
}


class RuleBasedCorrector:
    """Rule-based OCR text correction for Philippine forms."""

    def __init__(self):
        self.name_dict: set[str] = set()
        self.city_dict: set[str] = set()
        self.province_dict: set[str] = set()

    def load_dictionaries(self, names_file: str = None, cities_file: str = None):
        """Load lookup dictionaries for names and locations."""
        if names_file and Path(names_file).exists():
            with open(names_file) as f:
                self.name_dict = {line.strip().upper() for line in f if line.strip()}
        if cities_file and Path(cities_file).exists():
            with open(cities_file) as f:
                self.city_dict = {line.strip().upper() for line in f if line.strip()}

    def correct_text(self, text: str, field_type: str = "text") -> str:
        """Apply rule-based corrections to OCR text."""
        corrected = text

        # 1. Basic cleanup
        corrected = self._clean_whitespace(corrected)

        # 2. Field-specific corrections
        if field_type == "date":
            corrected = self._correct_date(corrected)
        elif field_type == "phone":
            corrected = self._correct_phone(corrected)
        elif field_type == "tin":
            corrected = self._correct_tin(corrected)
        elif field_type == "name":
            corrected = self._correct_name(corrected)
        elif field_type == "address":
            corrected = self._correct_address(corrected)

        # 3. PH-specific abbreviations
        for wrong, right in PH_CORRECTIONS.items():
            corrected = corrected.replace(wrong, right)

        return corrected

    def _clean_whitespace(self, text: str) -> str:
        """Normalize whitespace."""
        text = re.sub(r"\s+", " ", text).strip()
        # Remove spaces before punctuation
        text = re.sub(r"\s+([.,;:])", r"\1", text)
        return text

    def _correct_date(self, text: str) -> str:
        """Correct date formats."""
        # Common OCR errors in dates
        text = text.replace("O", "0")  # Letter O → digit 0
        text = text.replace("l", "1")  # Lowercase L → digit 1
        text = text.replace("I", "1")  # Capital I → digit 1

        # Normalize separators
        text = re.sub(r"[/\-\.]", "/", text)

        # Try to parse as MM/DD/YYYY
        match = re.match(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)
        if match:
            m, d, y = match.groups()
            if len(y) == 2:
                y = "20" + y if int(y) < 50 else "19" + y
            text = f"{int(m):02d}/{int(d):02d}/{y}"

        return text

    def _correct_phone(self, text: str) -> str:
        """Correct phone number formats."""
        # Keep only digits, +, -, (, )
        text = re.sub(r"[^\d+\-()\s]", "", text)
        # Letter-to-digit substitutions
        for letter, digit in [("O", "0"), ("l", "1"), ("I", "1"), ("S", "5"), ("B", "8")]:
            text = text.replace(letter, digit)
        return text

    def _correct_tin(self, text: str) -> str:
        """Correct TIN (Tax Identification Number) format: XXX-XXX-XXX-XXX."""
        # Remove non-digits and dashes
        digits = re.sub(r"[^\d]", "", text)
        if len(digits) == 12:
            return f"{digits[:3]}-{digits[3:6]}-{digits[6:9]}-{digits[9:12]}"
        elif len(digits) == 9:
            return f"{digits[:3]}-{digits[3:6]}-{digits[6:9]}"
        return text

    def _correct_name(self, text: str) -> str:
        """Correct name fields with title case and dictionary lookup."""
        words = text.split()
        corrected_words = []
        for word in words:
            # Title case
            word_upper = word.upper()
            if self.name_dict and word_upper in self.name_dict:
                corrected_words.append(word.title())
            elif word.isalpha():
                corrected_words.append(word.title())
            else:
                corrected_words.append(word)
        return " ".join(corrected_words)

    def _correct_address(self, text: str) -> str:
        """Correct address fields with abbreviation standardization."""
        for wrong, right in PH_CORRECTIONS.items():
            text = re.sub(rf"\b{re.escape(wrong)}\b", right, text, flags=re.IGNORECASE)
        return text


# Initialize corrector
corrector = RuleBasedCorrector()

# Test corrections
test_cases = [
    ("JUAN DELA CRUZ", "name"),
    ("brgy san antonio", "address"),
    ("O3/15/2O24", "date"),
    ("O9l7-123-4567", "phone"),
    ("123456789012", "tin"),
    ("  extra   spaces  here  ", "text"),
]

print("Rule-Based Correction Examples:")
print(f"{'Input':30s} | {'Type':10s} | {'Corrected':30s}")
print("-" * 75)
for text, field_type in test_cases:
    corrected = corrector.correct_text(text, field_type)
    print(f"{text:30s} | {field_type:10s} | {corrected:30s}")

In [ ]:
# ============================================================
# Step 5: Philippine Name & Address Dictionaries
# ============================================================
# Build lookup dictionaries for common Filipino names and PH locations.

# Common Filipino first names (sample — extend with full dataset)
FILIPINO_FIRST_NAMES = {
    "JUAN", "MARIA", "JOSE", "PEDRO", "ROSA", "CARLOS", "ANA",
    "ANTONIO", "FRANCISCO", "MANUEL", "LOURDES", "ELENA", "GABRIEL",
    "CHRISTOPHER", "MICHAEL", "PRINCESS", "ANGEL", "MARK", "JOHN",
    "MARY", "GRACE", "JOY", "FAITH", "HOPE", "LOVE", "DIVINE",
    "ROMEO", "JULIET", "BENJAMIN", "DIEGO", "ISABELLA", "SOPHIA",
    "ALTHEA", "DENISE", "MARIE", "ROSE", "MAE", "LYN", "ANN",
}

# Common Filipino surnames
FILIPINO_SURNAMES = {
    "SANTOS", "REYES", "CRUZ", "GARCIA", "DELA CRUZ", "GONZALES",
    "RAMOS", "AQUINO", "BAUTISTA", "MENDOZA", "TORRES", "FERNANDEZ",
    "VILLANUEVA", "CASTILLO", "RIVERA", "HERNANDEZ", "SANTIAGO",
    "MARTINEZ", "FLORES", "LOPEZ", "MORALES", "NAVARRO", "PEREZ",
    "DIZON", "ESPIRITU", "MANALO", "SORIANO", "PASCUAL", "ENRIQUEZ",
}

# Philippine regions/provinces (sample)
PH_PROVINCES = {
    "METRO MANILA", "CEBU", "DAVAO", "BULACAN", "PAMPANGA",
    "PANGASINAN", "LAGUNA", "BATANGAS", "CAVITE", "RIZAL",
    "NUEVA ECIJA", "TARLAC", "ZAMBALES", "ILOCOS NORTE",
    "ILOCOS SUR", "BENGUET", "LEYTE", "NEGROS OCCIDENTAL",
}

# Load dictionaries into corrector
corrector.name_dict = FILIPINO_FIRST_NAMES | FILIPINO_SURNAMES


def fuzzy_dict_match(word: str, dictionary: set[str], max_dist: int = 2) -> str | None:
    """Find the closest match in a dictionary using edit distance."""
    word_upper = word.upper()
    if word_upper in dictionary:
        return word_upper

    best_match = None
    best_dist = max_dist + 1

    for entry in dictionary:
        dist = editdistance.eval(word_upper, entry)
        if dist < best_dist:
            best_dist = dist
            best_match = entry

    return best_match if best_dist <= max_dist else None


# Test fuzzy matching
test_names = ["JUAM", "MRIA", "SANTOZ", "CRUX", "GONZALEZ", "BAUITISTA"]
print("Fuzzy Name Matching:")
for name in test_names:
    match = fuzzy_dict_match(name, FILIPINO_FIRST_NAMES | FILIPINO_SURNAMES)
    print(f"  '{name}' → '{match}'")

In [ ]:
# ============================================================
# Step 6: T5 Text Correction Model
# ============================================================
# Train a T5 model for sequence-to-sequence OCR error correction.
# Input: "correct: <ocr_text>"
# Output: "<corrected_text>"

import torch
from torch.utils.data import Dataset, DataLoader


class OCRCorrectionDataset(Dataset):
    """Dataset for T5 OCR text correction."""

    def __init__(self, pairs: list[tuple[str, str]], tokenizer, max_length: int = 128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ocr_text, correct_text = self.pairs[idx]

        # T5 input format
        input_text = f"correct: {ocr_text}"

        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        target_encoding = self.tokenizer(
            correct_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = target_encoding["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_encoding["input_ids"].squeeze(),
            "attention_mask": input_encoding["attention_mask"].squeeze(),
            "labels": labels,
        }


# Build training pairs from ground truth
# Uses ocr_value→verified_value pairs when available
# Falls back to noting that synthetic data will be the primary source
correction_pairs = []
for entry in ground_truth["entries"]:
    for field in entry["fields"]:
        ocr_val = field.get("ocr_value", "")
        gt_val = field.get("verified_value", "")
        # Only include pairs where OCR differs from ground truth
        if ocr_val and gt_val and ocr_val != gt_val:
            correction_pairs.append((ocr_val, gt_val))

print(f"Real correction pairs found: {len(correction_pairs)}")
if len(correction_pairs) == 0:
    print("No ocr_value in ground truth — training will use synthetic errors only.")
    print("For real OCR error pairs, export data from the API using notebook 01.")
else:
    print(f"\nSample pairs:")
    for ocr, gt in correction_pairs[:5]:
        print(f"  '{ocr}' → '{gt}'")

In [ ]:
# ============================================================
# Step 7: Augment Correction Training Data
# ============================================================
# Since we have limited real correction pairs, we generate synthetic ones.

import random


def generate_synthetic_errors(text: str, n_variants: int = 5) -> list[str]:
    """Generate synthetic OCR errors for a correct text string."""
    variants = []

    for _ in range(n_variants):
        chars = list(text)
        n_errors = random.randint(1, max(1, len(chars) // 5))

        for _ in range(n_errors):
            error_type = random.choice(["sub", "insert", "delete", "swap", "case"])
            pos = random.randint(0, max(0, len(chars) - 1))

            if error_type == "sub" and pos < len(chars):
                # Character substitution (common OCR confusions)
                confusions = {
                    "a": "o", "o": "a", "e": "c", "c": "e",
                    "n": "m", "m": "n", "i": "l", "l": "i",
                    "u": "v", "v": "u", "d": "cl", "w": "vv",
                    "0": "O", "1": "l", "5": "S", "8": "B",
                }
                char = chars[pos].lower()
                if char in confusions:
                    chars[pos] = confusions[char]
                else:
                    chars[pos] = random.choice("abcdefghijklmnopqrstuvwxyz")

            elif error_type == "insert":
                chars.insert(pos, random.choice("abcdefghijklmnopqrstuvwxyz "))

            elif error_type == "delete" and len(chars) > 1:
                chars.pop(pos)

            elif error_type == "swap" and pos < len(chars) - 1:
                chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]

            elif error_type == "case" and chars[pos].isalpha():
                chars[pos] = chars[pos].swapcase()

        variant = "".join(chars)
        if variant != text:  # Only keep actual errors
            variants.append(variant)

    return variants


# Generate synthetic training pairs
synthetic_pairs = []
for entry in ground_truth["entries"]:
    for field in entry["fields"]:
        gt_val = field.get("verified_value", "")
        if gt_val and field["type"] == "text":
            errors = generate_synthetic_errors(gt_val, n_variants=5)
            for error in errors:
                synthetic_pairs.append((error, gt_val))

# Combine real + synthetic pairs
all_pairs = correction_pairs + synthetic_pairs
random.shuffle(all_pairs)

print(f"Real pairs: {len(correction_pairs)}")
print(f"Synthetic pairs: {len(synthetic_pairs)}")
print(f"Total training pairs: {len(all_pairs)}")
print(f"\nSample synthetic errors:")
for ocr, gt in synthetic_pairs[:5]:
    print(f"  '{ocr}' → '{gt}'")

In [ ]:
# ============================================================
# Step 8: Train T5 Correction Model
# ============================================================
from transformers import T5ForConditionalGeneration, T5Tokenizer, TrainingArguments, Trainer

# Configuration
T5_MODEL = "t5-small"  # Use t5-base for better accuracy if GPU allows
T5_EPOCHS = 20
T5_BATCH = 16
T5_LR = 3e-4

# Load tokenizer and model
t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_model = t5_model.to(device)

print(f"T5 model: {T5_MODEL}")
print(f"Parameters: {sum(p.numel() for p in t5_model.parameters()):,}")
print(f"Device: {device}")

# Split data
split_idx = int(len(all_pairs) * 0.85)
train_pairs = all_pairs[:split_idx]
val_pairs = all_pairs[split_idx:]

# Create datasets
train_dataset = OCRCorrectionDataset(train_pairs, t5_tokenizer)
val_dataset = OCRCorrectionDataset(val_pairs, t5_tokenizer)

print(f"\nTrain: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Training
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "t5_checkpoints"),
    num_train_epochs=T5_EPOCHS,
    per_device_train_batch_size=T5_BATCH,
    per_device_eval_batch_size=T5_BATCH,
    learning_rate=T5_LR,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=t5_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

if len(train_dataset) > 0:
    print(f"Training T5 ({T5_MODEL}) for {T5_EPOCHS} epochs...")
    trainer.train()

    # Save model
    model_save_path = OUTPUT_DIR / "t5_corrector"
    trainer.save_model(str(model_save_path))
    t5_tokenizer.save_pretrained(str(model_save_path))
    print(f"\nModel saved to: {model_save_path}")
else:
    print("No training data. Generate pairs first.")

In [ ]:
# ============================================================
# Step 9: Combined Correction Pipeline
# ============================================================

class PostOCRCorrectionPipeline:
    """Combined correction pipeline: rules + dictionary + T5 model."""

    def __init__(
        self,
        rule_corrector: RuleBasedCorrector,
        t5_model_dir: str = None,
    ):
        self.rule_corrector = rule_corrector
        self.t5_model = None
        self.t5_tokenizer = None

        if t5_model_dir and Path(t5_model_dir).exists():
            self.t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_dir)
            self.t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_dir)
            self.t5_model.eval()
            self.t5_model = self.t5_model.to(device)
            print(f"T5 correction model loaded from {t5_model_dir}")

    def correct(
        self,
        text: str,
        field_type: str = "text",
        confidence: float = 1.0,
    ) -> dict:
        """Apply correction pipeline.

        For high-confidence text, only apply rules.
        For low-confidence text, also run T5 model.

        Returns: {corrected_text, method, changes_made}
        """
        original = text

        # Step 1: Always apply rule-based corrections
        rule_corrected = self.rule_corrector.correct_text(text, field_type)

        # Step 2: Dictionary fuzzy matching for names
        dict_corrected = rule_corrected
        if field_type == "name":
            words = rule_corrected.split()
            corrected_words = []
            for word in words:
                match = fuzzy_dict_match(
                    word,
                    FILIPINO_FIRST_NAMES | FILIPINO_SURNAMES,
                    max_dist=1,
                )
                if match:
                    corrected_words.append(match.title())
                else:
                    corrected_words.append(word)
            dict_corrected = " ".join(corrected_words)

        # Step 3: T5 model for low-confidence or complex corrections
        t5_corrected = dict_corrected
        method = "rules+dict"

        if self.t5_model and confidence < 0.85:
            input_text = f"correct: {dict_corrected}"
            inputs = self.t5_tokenizer(
                input_text, return_tensors="pt", max_length=128, truncation=True
            ).to(device)

            with torch.no_grad():
                outputs = self.t5_model.generate(
                    **inputs, max_length=128, num_beams=4, early_stopping=True
                )
            t5_corrected = self.t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
            method = "rules+dict+t5"

        return {
            "original": original,
            "corrected": t5_corrected,
            "method": method,
            "changed": original != t5_corrected,
        }


# Initialize pipeline
t5_model_path = OUTPUT_DIR / "t5_corrector"
pipeline = PostOCRCorrectionPipeline(
    rule_corrector=corrector,
    t5_model_dir=str(t5_model_path) if t5_model_path.exists() else None,
)

# Test the pipeline
test_inputs = [
    ("JUAM DELA CRUX", "name", 0.6),
    ("brgy san antonlo", "address", 0.7),
    ("O3/l5/2O24", "date", 0.5),
    ("SANTOS", "name", 0.95),
    ("MAKATI CITY", "address", 0.92),
]

print(f"{'Input':25s} | {'Type':8s} | {'Conf':5s} | {'Corrected':25s} | Method")
print("-" * 90)
for text, field_type, conf in test_inputs:
    result = pipeline.correct(text, field_type, conf)
    print(
        f"{text:25s} | {field_type:8s} | {conf:.2f}  | "
        f"{result['corrected']:25s} | {result['method']}"
    )

In [ ]:
# ============================================================
# Step 10: Evaluate Correction Pipeline
# ============================================================

def evaluate_correction_pipeline(
    pipeline: PostOCRCorrectionPipeline,
    test_pairs: list[tuple[str, str, str, float]],
) -> dict:
    """Evaluate the correction pipeline.

    test_pairs: list of (ocr_text, verified_text, field_type, confidence)
    """
    results = []
    before_cer = 0
    after_cer = 0
    improved = 0
    degraded = 0
    unchanged = 0

    for ocr_text, gt_text, field_type, conf in test_pairs:
        result = pipeline.correct(ocr_text, field_type, conf)
        corrected = result["corrected"]

        cer_before = editdistance.eval(ocr_text, gt_text) / max(len(gt_text), 1)
        cer_after = editdistance.eval(corrected, gt_text) / max(len(gt_text), 1)

        before_cer += cer_before
        after_cer += cer_after

        if cer_after < cer_before:
            improved += 1
        elif cer_after > cer_before:
            degraded += 1
        else:
            unchanged += 1

        results.append({
            "ocr": ocr_text,
            "corrected": corrected,
            "gt": gt_text,
            "cer_before": cer_before,
            "cer_after": cer_after,
        })

    n = len(test_pairs)
    return {
        "n_samples": n,
        "avg_cer_before": round(before_cer / n * 100, 2) if n else 0,
        "avg_cer_after": round(after_cer / n * 100, 2) if n else 0,
        "improved": improved,
        "degraded": degraded,
        "unchanged": unchanged,
        "improvement_rate": round(improved / n * 100, 2) if n else 0,
        "details": results,
    }


# Build test set from ground truth
# If no ocr_value available, generate synthetic errors for evaluation
test_set = []
for entry in ground_truth["entries"]:
    for field in entry["fields"]:
        gt_val = field.get("verified_value", "")
        ocr_val = field.get("ocr_value", "")
        if not gt_val:
            continue
        if ocr_val and ocr_val != gt_val:
            # Real OCR error pair
            test_set.append((ocr_val, gt_val, field["type"], 0.7))
        elif field["type"] == "text" and len(gt_val) > 2:
            # Generate one synthetic error for testing
            errors = generate_synthetic_errors(gt_val, n_variants=1)
            if errors:
                test_set.append((errors[0], gt_val, field["type"], 0.7))

if test_set:
    eval_results = evaluate_correction_pipeline(pipeline, test_set)

    print(f"{'='*60}")
    print("CORRECTION PIPELINE EVALUATION")
    print(f"{'='*60}")
    print(f"  Samples:         {eval_results['n_samples']}")
    print(f"  CER Before:      {eval_results['avg_cer_before']}%")
    print(f"  CER After:       {eval_results['avg_cer_after']}%")
    print(f"  Improved:        {eval_results['improved']}")
    print(f"  Degraded:        {eval_results['degraded']}")
    print(f"  Unchanged:       {eval_results['unchanged']}")
    print(f"  Improvement Rate:{eval_results['improvement_rate']}%")
else:
    print("No test data available.")

In [ ]:
# ============================================================
# Step 11: Export for Integration
# ============================================================

# Save the complete correction config
correction_config = {
    "rule_corrections": PH_CORRECTIONS,
    "ocr_confusion_map": OCR_CONFUSION_MAP,
    "name_dictionary_size": len(FILIPINO_FIRST_NAMES) + len(FILIPINO_SURNAMES),
    "t5_model": str(t5_model_path) if t5_model_path.exists() else None,
    "confidence_threshold": 0.85,
    "integration_notes": (
        "Add _post_correct() method to ocr_service.py. "
        "Call after _map_fields_with_ai(). "
        "For fields with confidence < 0.85, run T5 correction. "
        "Always apply rule-based corrections. "
        "Uses RuleBasedCorrector class and PostOCRCorrectionPipeline."
    ),
}

with open(OUTPUT_DIR / "correction_config.json", "w") as f:
    json.dump(correction_config, f, indent=2, default=str)

# Save dictionaries as files for the backend
with open(OUTPUT_DIR / "filipino_names.txt", "w") as f:
    for name in sorted(FILIPINO_FIRST_NAMES | FILIPINO_SURNAMES):
        f.write(name + "\n")

with open(OUTPUT_DIR / "ph_provinces.txt", "w") as f:
    for prov in sorted(PH_PROVINCES):
        f.write(prov + "\n")

print(f"Correction config saved to: {OUTPUT_DIR / 'correction_config.json'}")
print(f"Name dictionary saved to: {OUTPUT_DIR / 'filipino_names.txt'}")
print(f"Province dictionary saved to: {OUTPUT_DIR / 'ph_provinces.txt'}")
print(f"\n=== Integration Steps ===")
print(f"1. Copy correction files to backend/app/services/data/")
print(f"2. Add RuleBasedCorrector class to ocr_service.py")
print(f"3. Add _post_correct() method after field mapping")
print(f"4. If T5 model available, copy to backend/app/models/ml/t5_corrector/")
print(f"5. Test end-to-end OCR → mapping → correction pipeline")

---

# Section 7: Evaluation & Benchmark

Evaluate and benchmark the full OCR pipeline — compute CER, WER, field accuracy per template.

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Option A: Load from exported dataset (run 01_data_export first)
DATASET_PATH = "./training_data/labels.json"

# Option B: Fetch directly from API
USE_API = False  # Set True to fetch from API instead of local file
API_BASE_URL = "http://localhost:8000/api/v1"
ADMIN_EMAIL = "admin@smartform.local"
ADMIN_PASSWORD = "admin123"

In [ ]:
import json
import os
from pathlib import Path

import editdistance
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (12, 6)
matplotlib.rcParams["font.size"] = 11

## Step 1: Load Data

In [ ]:
def load_from_file(path: str) -> list[dict]:
    """Load training data from file. Falls back to ground_truth.json."""
    from pathlib import Path

    if not Path(path).exists():
        fallback = GROUND_TRUTH_FALLBACK if 'GROUND_TRUTH_FALLBACK' in dir() else './sample_data/ground_truth.json'
        if Path(fallback).exists():
            print(f"Primary file not found: {path}")
            print(f"Using fallback: {fallback}")
            with open(fallback) as f:
                gt = json.load(f)
            # Convert ground_truth format to training data format
            records = []
            for entry in gt.get('entries', []):
                for field in entry.get('fields', []):
                    records.append({
                        'field_name': field.get('field_name', ''),
                        'ocr_value': field.get('ocr_value', field.get('verified_value', '')),
                        'verified_value': field.get('verified_value', ''),
                        'confidence': field.get('confidence', 0.0),
                        'was_corrected': field.get('was_corrected', False),
                        'field_type': field.get('type', 'text'),
                        'template_name': entry.get('template_name', 'unknown'),
                    })
            return records
        else:
            raise FileNotFoundError(f"Neither {path} nor {fallback} found. Run notebook 01 first.")

    with open(path) as f:
        data = json.load(f)
    return data.get('entries', data) if isinstance(data, dict) else data


## Step 2: Compute Error Metrics

In [ ]:
def compute_cer(ocr_value: str, verified_value: str) -> float:
    """Character Error Rate: edit_distance / len(reference)."""
    if not verified_value:
        return 0.0 if not ocr_value else 1.0
    return editdistance.eval(ocr_value, verified_value) / len(verified_value)


def compute_wer(ocr_value: str, verified_value: str) -> float:
    """Word Error Rate: edit_distance on word tokens / len(reference words)."""
    ref_words = verified_value.split()
    hyp_words = ocr_value.split()
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    return editdistance.eval(hyp_words, ref_words) / len(ref_words)


def normalize_value(v: str | None) -> str:
    """Normalize a value for comparison (strip, lowercase)."""
    if v is None:
        return ""
    return v.strip()


# Build a flat list of field-level evaluation records
records = []

for entry in entries:
    template_name = entry.get("template_name", "Unknown")

    for field in entry.get("fields", []):
        ocr_val = normalize_value(field.get("ocr_value"))
        verified_val = normalize_value(field.get("verified_value"))

        # Skip fields with no verified value (no ground truth)
        if not verified_val:
            continue

        exact_match = ocr_val == verified_val
        exact_match_ci = ocr_val.lower() == verified_val.lower()
        cer = compute_cer(ocr_val, verified_val)
        wer = compute_wer(ocr_val, verified_val)

        records.append({
            "entry_id": entry.get("entry_id"),
            "template_name": template_name,
            "field_name": field["field_name"],
            "ocr_value": ocr_val,
            "verified_value": verified_val,
            "confidence": field.get("confidence", 0.0),
            "was_corrected": field.get("was_corrected", False),
            "exact_match": exact_match,
            "exact_match_ci": exact_match_ci,
            "cer": cer,
            "wer": wer,
        })

df = pd.DataFrame(records)
print(f"Total evaluable fields: {len(df)}")
df.head(10)

## Step 3: Overall Accuracy Report

In [ ]:
print("=" * 70)
print("OVERALL OCR PIPELINE EVALUATION")
print("=" * 70)

if len(df) > 0:
    print(f"Total entries evaluated:     {df['entry_id'].nunique()}")
    print(f"Total fields evaluated:      {len(df)}")
    print()
    print(f"Field Accuracy (exact):      {df['exact_match'].mean() * 100:.2f}%")
    print(f"Field Accuracy (case-ins.):  {df['exact_match_ci'].mean() * 100:.2f}%")
    print(f"Character Error Rate (CER):  {df['cer'].mean() * 100:.2f}%")
    print(f"Word Error Rate (WER):       {df['wer'].mean() * 100:.2f}%")
    print(f"Correction Rate:             {df['was_corrected'].mean() * 100:.2f}%")
    print(f"Average OCR Confidence:      {df['confidence'].mean():.4f}")
    print()

    # Accuracy targets (from project plan)
    targets = {
        "Field Accuracy": 85.0,
        "CER": 5.0,
        "WER": 10.0,
        "Correction Rate": 15.0,
    }
    actuals = {
        "Field Accuracy": df['exact_match'].mean() * 100,
        "CER": df['cer'].mean() * 100,
        "WER": df['wer'].mean() * 100,
        "Correction Rate": df['was_corrected'].mean() * 100,
    }

    print("TARGET vs ACTUAL:")
    for metric, target in targets.items():
        actual = actuals[metric]
        if metric == "Field Accuracy":
            status = "PASS" if actual >= target else "FAIL"
        else:
            status = "PASS" if actual <= target else "FAIL"
        symbol = "\u2705" if status == "PASS" else "\u274c"
        print(f"  {symbol} {metric}: {actual:.2f}% (target: {'>' if metric == 'Field Accuracy' else '<'} {target}%)")
else:
    print("No evaluable fields found — verify forms first!")

## Step 4: Per-Template Breakdown

In [ ]:
if len(df) > 0:
    template_stats = (
        df.groupby("template_name")
        .agg(
            field_count=("field_name", "count"),
            accuracy=("exact_match", "mean"),
            accuracy_ci=("exact_match_ci", "mean"),
            avg_cer=("cer", "mean"),
            avg_wer=("wer", "mean"),
            correction_rate=("was_corrected", "mean"),
            avg_confidence=("confidence", "mean"),
        )
        .round(4)
    )

    # Format percentages
    for col in ["accuracy", "accuracy_ci", "avg_cer", "avg_wer", "correction_rate"]:
        template_stats[col] = (template_stats[col] * 100).round(2)

    template_stats.columns = [
        "Fields", "Accuracy %", "Accuracy (CI) %", "CER %", "WER %",
        "Correction %", "Avg Confidence"
    ]

    print("\nPER-TEMPLATE ACCURACY:")
    print(template_stats.to_string())

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    template_stats["Accuracy %"].plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Field Accuracy by Template")
    axes[0].set_xlabel("Accuracy (%)")
    axes[0].axvline(x=85, color="red", linestyle="--", label="Target (85%)")
    axes[0].legend()

    template_stats["CER %"].plot(kind="barh", ax=axes[1], color="coral")
    axes[1].set_title("Character Error Rate by Template")
    axes[1].set_xlabel("CER (%)")
    axes[1].axvline(x=5, color="red", linestyle="--", label="Target (<5%)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("template_accuracy.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 5: Per-Field Breakdown (Most Problematic Fields)

In [ ]:
if len(df) > 0:
    field_stats = (
        df.groupby("field_name")
        .agg(
            count=("field_name", "count"),
            accuracy=("exact_match", "mean"),
            avg_cer=("cer", "mean"),
            correction_rate=("was_corrected", "mean"),
            avg_confidence=("confidence", "mean"),
        )
        .round(4)
    )

    for col in ["accuracy", "avg_cer", "correction_rate"]:
        field_stats[col] = (field_stats[col] * 100).round(2)

    # Sort by worst accuracy
    field_stats = field_stats.sort_values("accuracy")

    field_stats.columns = ["Count", "Accuracy %", "CER %", "Correction %", "Avg Confidence"]

    print("\nFIELD-LEVEL ACCURACY (worst to best):")
    print(field_stats.to_string())

    # Top 15 worst fields
    worst = field_stats.head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    worst["Accuracy %"].plot(kind="barh", ax=ax, color="tomato")
    ax.set_title("15 Most Problematic Fields (Lowest Accuracy)")
    ax.set_xlabel("Accuracy (%)")
    ax.axvline(x=85, color="green", linestyle="--", label="Target (85%)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("field_accuracy.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 6: Confidence Calibration Analysis

In [ ]:
if len(df) > 0:
    # Bin confidence scores and compare to actual accuracy
    bins = [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.01]
    labels = ["0-0.3", "0.3-0.5", "0.5-0.7", "0.7-0.8", "0.8-0.9", "0.9-0.95", "0.95-1.0"]
    df["conf_bin"] = pd.cut(df["confidence"], bins=bins, labels=labels, right=False)

    calibration = (
        df.groupby("conf_bin", observed=True)
        .agg(
            count=("field_name", "count"),
            actual_accuracy=("exact_match", "mean"),
            avg_confidence=("confidence", "mean"),
        )
        .round(4)
    )

    print("\nCONFIDENCE CALIBRATION:")
    print("(Is the model's confidence score well-calibrated with actual accuracy?)")
    print(calibration.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(calibration))
    ax.bar(x, calibration["actual_accuracy"] * 100, alpha=0.7, label="Actual Accuracy")
    ax.bar(x, calibration["avg_confidence"] * 100, alpha=0.4, label="Reported Confidence")
    ax.set_xticks(x)
    ax.set_xticklabels(calibration.index, rotation=45)
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Confidence Calibration: Reported vs Actual Accuracy")
    ax.legend()
    plt.tight_layout()
    plt.savefig("confidence_calibration.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 7: Error Analysis — Common OCR Mistakes

In [ ]:
if len(df) > 0:
    # Show examples where OCR was wrong
    errors_df = df[~df["exact_match"] & (df["ocr_value"] != "")].copy()
    errors_df = errors_df.sort_values("cer", ascending=False)

    print(f"\nERROR ANALYSIS ({len(errors_df)} incorrect fields):")
    print("=" * 80)

    # Categorize error types
    def classify_error(row):
        ocr = row["ocr_value"]
        verified = row["verified_value"]

        if ocr.lower() == verified.lower():
            return "case_only"
        if ocr.replace(" ", "") == verified.replace(" ", ""):
            return "spacing_only"
        if ocr in verified or verified in ocr:
            return "partial_match"
        if len(ocr) == 0:
            return "empty_ocr"
        return "significant_error"


    if len(errors_df) == 0:
        print("No OCR errors found — all fields matched exactly!")
    else:
        errors_df["error_type"] = errors_df.apply(classify_error, axis=1)

        error_dist = errors_df["error_type"].value_counts()
        print("\nError type distribution:")
        for etype, count in error_dist.items():
            pct = count / len(errors_df) * 100
            print(f"  {etype}: {count} ({pct:.1f}%)")

        print("\nTop 20 worst OCR errors (by CER):")
        worst_errors = errors_df.head(20)[
            ["template_name", "field_name", "ocr_value", "verified_value", "cer", "confidence"]
        ]
        for _, row in worst_errors.iterrows():
            print(f"  [{row['template_name']}] {row['field_name']}:")
            print(f"    OCR:      '{row['ocr_value']}'")
            print(f"    Verified: '{row['verified_value']}'")
            print(f"    CER: {row['cer']:.2%} | Confidence: {row['confidence']:.2f}")
            print()

## Step 8: Save Baseline Report

In [ ]:
if len(df) > 0:
    from datetime import datetime

    baseline = {
        "evaluation_date": datetime.utcnow().isoformat(),
        "pipeline_version": "baseline (pre-ML)",
        "total_entries": int(df["entry_id"].nunique()),
        "total_fields": len(df),
        "overall": {
            "field_accuracy": round(df["exact_match"].mean() * 100, 2),
            "field_accuracy_ci": round(df["exact_match_ci"].mean() * 100, 2),
            "cer": round(df["cer"].mean() * 100, 2),
            "wer": round(df["wer"].mean() * 100, 2),
            "correction_rate": round(df["was_corrected"].mean() * 100, 2),
            "avg_confidence": round(df["confidence"].mean(), 4),
        },
        "per_template": template_stats.reset_index().to_dict(orient="records"),
        "worst_fields": field_stats.head(10).reset_index().to_dict(orient="records"),
        "targets": {
            "field_accuracy": "> 85%",
            "cer": "< 5%",
            "wer": "< 10%",
            "correction_rate": "< 15%",
        },
    }

    output_path = Path("./evaluation_results")
    output_path.mkdir(exist_ok=True)

    report_path = output_path / f"baseline_{datetime.utcnow().strftime('%Y%m%d')}.json"
    with open(report_path, "w") as f:
        json.dump(baseline, f, indent=2, default=str)

    print(f"\nBaseline report saved to: {report_path}")
    print("\nUse this as the reference point to measure improvement after ML training.")
    print("Re-run this notebook after each training cycle to track progress.")
else:
    print("No data to save.")

## Step 9: Comparison (Run After Training)

After training models and deploying them, re-run this notebook and compare:
- Load the baseline report
- Run evaluation on new data processed by the updated pipeline
- Show delta improvements

In [ ]:
def compare_baselines(baseline_path: str, current_metrics: dict) -> None:
    """Compare current metrics against a saved baseline."""
    with open(baseline_path, "r") as f:
        baseline = json.load(f)

    print("=" * 70)
    print(f"COMPARISON: {baseline['pipeline_version']} vs Current")
    print("=" * 70)

    for metric in ["field_accuracy", "cer", "wer", "correction_rate"]:
        old_val = baseline["overall"][metric]
        new_val = current_metrics[metric]
        delta = new_val - old_val

        if metric == "field_accuracy":
            improved = delta > 0
        else:
            improved = delta < 0

        symbol = "\u2b06\ufe0f" if improved else "\u2b07\ufe0f"
        print(f"  {symbol} {metric}: {old_val:.2f}% -> {new_val:.2f}% ({delta:+.2f}%)")


# Example usage (uncomment after training):
# Uncomment after first evaluation run:
# compare_baselines(
#     "./evaluation_results/baseline_20260305.json",
#     {
#         "field_accuracy": df["exact_match"].mean() * 100,
#         "cer": df["cer"].mean() * 100,
#         "wer": df["wer"].mean() * 100,
#         "correction_rate": df["was_corrected"].mean() * 100,
#     }
# )